In [ ]:

# ============================================================
# MDAS — NEW COLAB NOTEBOOK INITIALIZATION
# Continue from Step 40 without repeating previous experiments
# ============================================================

from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import json

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")

PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

STEP39 = PROJECT / "Step39_Original_Estimator_Numerical_Replay"
STEP40 = PROJECT / "Step40_Reproduction_Discrepancy_Audit"
STEP30 = PROJECT / "Step30_Frozen_Potsdam_Benchmark"

required = {
    "Original NCC replay":
        STEP39 / "Step39_NCC_Replay.csv",
    "Original phase replay":
        STEP39 / "Step39_Phase_Replay.csv",
    "Step 39 report":
        STEP39 / "Step39_Final_Report.json",
    "Potsdam benchmark":
        STEP30 / "Step30_Final_Report.json"
}

print("=" * 70)
print("MDAS — NEW NOTEBOOK INITIALIZATION")
print("=" * 70)

for name, path in required.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")
    print(f"VERIFIED: {name}")

report = json.loads(
    (STEP39 / "Step39_Final_Report.json").read_text()
)

print("\nPrevious experimental results:")
print("NCC reproduced:", report["ncc_matches"], "/ 680")
print("Phase reproduced:", report["phase_matches"], "/ 680")

step40_report = STEP40 / "Step40_Final_Report.json"

if step40_report.exists():
    print("\nStep 40 already completed:")
    print(json.loads(step40_report.read_text())["status"])
else:
    print("\nNEXT: Run Step 40 in this notebook.")

print("\nREADY — Previous results preserved.")


In [ ]:

# =============================================================================
# STEP 40 — HISTORICAL REPRODUCTION DISCREPANCY RESOLUTION
#
# Goal:
#   Resolve the two Step 39 phase-correlation discrepancies.
#
# Input:
#   Step39_NCC_Replay.csv
#   Step39_Phase_Replay.csv
#   Step39_Final_Report.json
#   Step20_Test_Registration.csv
#   Step20_Frozen_Method.json
#   Step13_Gated_Test_Registration.csv
#
# Checks:
#   - Exact saved-record consistency
#   - Errors at 1e-5 and 3e-5 pixel tolerances
#   - Detailed inspection of all mismatched cases
#   - Frozen phase bias and snapping decisions
#   - Whether small raw differences change EPE or identity decisions
#   - Original NCC source-replay completeness
#
# No estimator reruns.
# No parameter tuning or corrections.
# Original historical files remain unchanged.
# =============================================================================

from google.colab import drive
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

STEP13 = PROJECT / "Step13_Identity_Preserving_Gate"
STEP20 = PROJECT / "Step20_Bounded_Phase_Correlation"
STEP39 = PROJECT / "Step39_Original_Estimator_Numerical_Replay"

OUT = PROJECT / "Step40_Reproduction_Discrepancy_Audit"
OUT.mkdir(parents=True, exist_ok=True)

NCC_REPLAY = STEP39 / "Step39_NCC_Replay.csv"
PHASE_REPLAY = STEP39 / "Step39_Phase_Replay.csv"
STEP39_REPORT = STEP39 / "Step39_Final_Report.json"

PHASE_HISTORY = STEP20 / "Step20_Test_Registration.csv"
PHASE_CONFIG = STEP20 / "Step20_Frozen_Method.json"

GATE_HISTORY = STEP13 / "Step13_Gated_Test_Registration.csv"

FILES = {
    "ncc_replay": NCC_REPLAY,
    "phase_replay": PHASE_REPLAY,
    "step39_report": STEP39_REPORT,
    "phase_history": PHASE_HISTORY,
    "phase_config": PHASE_CONFIG,
    "gate_history": GATE_HISTORY,
}

OUTPUTS = [
    "Step40_All_Phase_Replay_Differences.csv",
    "Step40_Phase_Mismatch_Details.csv",
    "Step40_Tolerance_Sensitivity.csv",
    "Step40_Replayed_Phase_Decisions.csv",
    "Step40_Reproduction_Summary.csv",
    "Step40_Final_Report.json",
]

if any((OUT / filename).exists() for filename in OUTPUTS):
    raise FileExistsError(
        "Existing Step 40 results found. "
        "Preserve the previous output and choose a new directory."
    )

for name, path in FILES.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()

def key_frame(frame):
    result = frame.copy()

    result["image_key"] = (
        result["image"]
        .astype(str)
        .str.replace("\\", "/", regex=False)
        .str.split("/")
        .str[-1]
    )

    for name in ["true_dy", "true_dx"]:
        result[name] = pd.to_numeric(
            result[name], errors="raise"
        ).round(6)

    return result

KEY = ["image_key", "true_dy", "true_dx"]

print("=" * 88)
print("STEP 40 — MDAS ESTIMATOR REPRODUCTION AUDIT")
print("=" * 88)

ncc = key_frame(pd.read_csv(NCC_REPLAY))
phase = key_frame(pd.read_csv(PHASE_REPLAY))
history = key_frame(pd.read_csv(PHASE_HISTORY))
gate = key_frame(pd.read_csv(GATE_HISTORY))

config = json.loads(
    PHASE_CONFIG.read_text(encoding="utf-8")
)

prior = json.loads(
    STEP39_REPORT.read_text(encoding="utf-8")
)

assert len(ncc) == 680
assert len(phase) == 680
assert len(history) == 680
assert len(gate) == 680

for frame, label in [
    (ncc, "NCC"),
    (phase, "phase"),
    (history, "historical phase"),
    (gate, "Step13 gate")
]:
    if frame.duplicated(KEY).any():
        raise ValueError(
            f"Duplicate image/shift keys in {label}"
        )

assert prior["ncc_cases"] == 680
assert prior["phase_cases"] == 680

print("\n40A — VERIFIED INPUTS")
print("NCC replay cases:", len(ncc))
print("Phase replay cases:", len(phase))
print("Historical phase records:", len(history))
print("Saved NCC matches at 1e-5:", prior["ncc_matches"])
print("Saved phase matches at 1e-5:", prior["phase_matches"])

# -------------------------------------------------------------------------
# 40B — RECOMPUTE NUMERICAL DIFFERENCES
# -------------------------------------------------------------------------

ncc["recomputed_difference"] = np.hypot(
    ncc["replayed_dy"] - ncc["recorded_dy"],
    ncc["replayed_dx"] - ncc["recorded_dx"]
)

phase["recomputed_difference"] = np.hypot(
    phase["replayed_raw_dy"] - phase["recorded_raw_dy"],
    phase["replayed_raw_dx"] - phase["recorded_raw_dx"]
)

assert np.allclose(
    ncc["recomputed_difference"],
    ncc["difference_epe"],
    atol=1e-10,
    rtol=1e-5
)

assert np.allclose(
    phase["recomputed_difference"],
    phase["difference_epe"],
    atol=1e-10,
    rtol=1e-5
)

assert np.isfinite(
    ncc["recomputed_difference"]
).all()

assert np.isfinite(
    phase["recomputed_difference"]
).all()

strict_tol = 1e-5
review_tol = 3e-5

strict_ncc = (
    ncc["recomputed_difference"] <= strict_tol
)

strict_phase = (
    phase["recomputed_difference"] <= strict_tol
)

review_ncc = (
    ncc["recomputed_difference"] <= review_tol
)

review_phase = (
    phase["recomputed_difference"] <= review_tol
)

print("\n40B — REPRODUCTION COUNTS")
print(
    "NCC at 1e-5:",
    int(strict_ncc.sum()),
    "/ 680"
)
print(
    "Phase at 1e-5:",
    int(strict_phase.sum()),
    "/ 680"
)
print(
    "Phase at 3e-5:",
    int(review_phase.sum()),
    "/ 680"
)

# -------------------------------------------------------------------------
# 40C — RECONCILE AGAINST ACTUAL HISTORICAL PHASE CSV
# -------------------------------------------------------------------------

historical_fields = [
    "raw_dy", "raw_dx",
    "estimated_dy", "estimated_dx",
    "snapped", "epe"
]

merged = phase.merge(
    history[KEY + historical_fields],
    on=KEY,
    how="left",
    validate="one_to_one",
    indicator=True
)

if not merged["_merge"].eq("both").all():
    raise RuntimeError(
        "Step39 replay does not match all historical phase keys."
    )

assert np.allclose(
    merged["recorded_raw_dy"],
    merged["raw_dy"],
    rtol=0,
    atol=1e-9
)

assert np.allclose(
    merged["recorded_raw_dx"],
    merged["raw_dx"],
    rtol=0,
    atol=1e-9
)

# -------------------------------------------------------------------------
# 40D — APPLY FROZEN PHASE POSTPROCESSING TO BOTH
#        RECORDED AND REPLAYED RAW ESTIMATES
# -------------------------------------------------------------------------

bias_y = float(config["bias_dy"])
bias_x = float(config["bias_dx"])
tau = float(config["tau"])

assert int(config["radius"]) == 5
assert np.isclose(tau, 0.25)

def apply_phase_postprocessing(raw_y, raw_x):

    corrected_y = raw_y - bias_y
    corrected_x = raw_x - bias_x

    magnitude = np.hypot(
        corrected_y,
        corrected_x
    )

    snapped = magnitude < tau

    final_y = np.where(
        snapped, 0.0, corrected_y
    )

    final_x = np.where(
        snapped, 0.0, corrected_x
    )

    return final_y, final_x, snapped, magnitude

recorded_final_y, recorded_final_x, recorded_snap, recorded_mag = (
    apply_phase_postprocessing(
        merged["raw_dy"].to_numpy(float),
        merged["raw_dx"].to_numpy(float)
    )
)

replayed_final_y, replayed_final_x, replayed_snap, replayed_mag = (
    apply_phase_postprocessing(
        merged["replayed_raw_dy"].to_numpy(float),
        merged["replayed_raw_dx"].to_numpy(float)
    )
)

# Confirm previously established Step38 postprocessing.
assert np.allclose(
    recorded_final_y,
    merged["estimated_dy"],
    rtol=0,
    atol=1e-8
)

assert np.allclose(
    recorded_final_x,
    merged["estimated_dx"],
    rtol=0,
    atol=1e-8
)

historical_snap = (
    merged["snapped"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
    .to_numpy(bool)
)

assert np.array_equal(
    recorded_snap,
    historical_snap
)

raw_epe_difference = merged[
    "recomputed_difference"
].to_numpy(float)

final_displacement_difference = np.hypot(
    replayed_final_y - recorded_final_y,
    replayed_final_x - recorded_final_x
)

recorded_epe = np.hypot(
    recorded_final_y - merged["true_dy"].to_numpy(float),
    recorded_final_x - merged["true_dx"].to_numpy(float)
)

replayed_epe = np.hypot(
    replayed_final_y - merged["true_dy"].to_numpy(float),
    replayed_final_x - merged["true_dx"].to_numpy(float)
)

epe_change = np.abs(
    replayed_epe - recorded_epe
)

snap_changed = (
    replayed_snap != recorded_snap
)

decision_table = merged[
    KEY + [
        "recorded_raw_dy",
        "recorded_raw_dx",
        "replayed_raw_dy",
        "replayed_raw_dx"
    ]
].copy()

decision_table["raw_difference_px"] = raw_epe_difference
decision_table["recorded_corrected_magnitude"] = recorded_mag
decision_table["replayed_corrected_magnitude"] = replayed_mag
decision_table["recorded_snapped"] = recorded_snap
decision_table["replayed_snapped"] = replayed_snap
decision_table["snap_decision_changed"] = snap_changed
decision_table["final_displacement_difference_px"] = (
    final_displacement_difference
)
decision_table["recorded_epe"] = recorded_epe
decision_table["replayed_epe"] = replayed_epe
decision_table["absolute_epe_change"] = epe_change

decision_table.to_csv(
    OUT / "Step40_Replayed_Phase_Decisions.csv",
    index=False
)

phase.to_csv(
    OUT / "Step40_All_Phase_Replay_Differences.csv",
    index=False
)

# -------------------------------------------------------------------------
# 40E — INSPECT THE TWO STRICT-TOLERANCE FAILURES
# -------------------------------------------------------------------------

mismatch_mask = (
    raw_epe_difference > strict_tol
)

mismatches = decision_table.loc[
    mismatch_mask
].copy()

mismatches = mismatches.sort_values(
    "raw_difference_px",
    ascending=False
)

mismatches.to_csv(
    OUT / "Step40_Phase_Mismatch_Details.csv",
    index=False
)

print("\n40C — STRICT-TOLERANCE PHASE MISMATCHES")
print("Number:", len(mismatches))

if len(mismatches):
    print(
        mismatches[
            [
                "image_key",
                "true_dy",
                "true_dx",
                "raw_difference_px",
                "snap_decision_changed",
                "final_displacement_difference_px",
                "absolute_epe_change"
            ]
        ].to_string(index=False)
    )

print("\n40D — DECISION STABILITY")
print("Changed identity decisions:", int(snap_changed.sum()))
print(
    "Maximum final displacement difference:",
    float(final_displacement_difference.max())
)
print(
    "Maximum EPE difference:",
    float(epe_change.max())
)

# -------------------------------------------------------------------------
# 40F — TOLERANCE SENSITIVITY
# -------------------------------------------------------------------------

tolerance_values = [
    1e-6, 3e-6, 5e-6, 1e-5, 2e-5, 3e-5, 5e-5
]

tolerance_rows = []

for tolerance in tolerance_values:
    tolerance_rows.append({
        "tolerance_pixels": tolerance,
        "ncc_matches": int(
            (ncc["recomputed_difference"] <= tolerance).sum()
        ),
        "phase_matches": int(
            (phase["recomputed_difference"] <= tolerance).sum()
        ),
        "ncc_max_difference_px": float(
            ncc["recomputed_difference"].max()
        ),
        "phase_max_difference_px": float(
            phase["recomputed_difference"].max()
        )
    })

tolerance_df = pd.DataFrame(tolerance_rows)

tolerance_df.to_csv(
    OUT / "Step40_Tolerance_Sensitivity.csv",
    index=False
)

print("\n40E — TOLERANCE SENSITIVITY")
print(tolerance_df.to_string(index=False))

# -------------------------------------------------------------------------
# 40G — FINAL AUDIT
# -------------------------------------------------------------------------

ncc_full_at_strict = bool(strict_ncc.all())
phase_full_at_review = bool(review_phase.all())
phase_decisions_stable = bool(
    not snap_changed.any()
)

# Pass criterion for agreement at the documented
# review tolerance, not bitwise reproduction.
reproduction_consistent = bool(
    ncc_full_at_strict
    and phase_full_at_review
    and phase_decisions_stable
)

if reproduction_consistent:
    status = "NUMERICAL_REPRODUCTION_CONSISTENT"
else:
    status = "REPRODUCTION_DISCREPANCIES_REMAIN"

summary = pd.DataFrame([
    {
        "method": "original_ncc",
        "cases": 680,
        "matches_at_1e5": int(strict_ncc.sum()),
        "matches_at_3e5": int(review_ncc.sum()),
        "mean_difference_px": float(
            ncc["recomputed_difference"].mean()
        ),
        "max_difference_px": float(
            ncc["recomputed_difference"].max()
        )
    },
    {
        "method": "original_phase_raw",
        "cases": 680,
        "matches_at_1e5": int(strict_phase.sum()),
        "matches_at_3e5": int(review_phase.sum()),
        "mean_difference_px": float(
            phase["recomputed_difference"].mean()
        ),
        "max_difference_px": float(
            phase["recomputed_difference"].max()
        )
    }
])

summary.to_csv(
    OUT / "Step40_Reproduction_Summary.csv",
    index=False
)

report = {
    "experiment": "Historical estimator numerical discrepancy audit",
    "status": status,
    "ncc_cases": 680,
    "phase_cases": 680,
    "ncc_matches_at_1e_minus_5": int(strict_ncc.sum()),
    "phase_matches_at_1e_minus_5": int(strict_phase.sum()),
    "ncc_matches_at_3e_minus_5": int(review_ncc.sum()),
    "phase_matches_at_3e_minus_5": int(review_phase.sum()),
    "ncc_max_difference_px": float(
        ncc["recomputed_difference"].max()
    ),
    "phase_max_difference_px": float(
        phase["recomputed_difference"].max()
    ),
    "phase_strict_mismatch_cases": int(
        mismatch_mask.sum()
    ),
    "phase_identity_snap_decisions_changed": int(
        snap_changed.sum()
    ),
    "phase_max_final_displacement_difference_px": float(
        final_displacement_difference.max()
    ),
    "phase_max_absolute_epe_change": float(
        epe_change.max()
    ),
    "original_parameters_changed": False,
    "original_estimators_rerun": False,
    "external_potsdam_evaluated": False,
    "source_step39_report_sha256": sha256(STEP39_REPORT),
    "source_ncc_replay_sha256": sha256(NCC_REPLAY),
    "source_phase_replay_sha256": sha256(PHASE_REPLAY),
    "limitations": [
        "The 3e-5 tolerance is a stated numerical agreement "
        "criterion, not proof of bitwise identical computation.",
        "Replays use the previously explored MDAS test mosaic.",
        "Stable snap decisions do not establish the same "
        "FFT surface in every numerical detail.",
        "Potsdam remains a single independent acquisition.",
        "No external transfer or fusion checkpoint testing "
        "is performed here."
    ],
    "output_directory": str(OUT)
}

(OUT / "Step40_Final_Report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 88)
print("STEP 40 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 40 COMPLETE")
print("No estimator reruns or parameter changes.")
print("Historical and external data unchanged.")


In [ ]:

# =============================================================================
# STEP 41C — FROZEN MDAS REGISTRATION TRANSFER TO POTSDAM
#             WITH SPECTRAL METADATA RECONCILIATION
#
# Changes from Step41B:
#   - Explicitly measure CSV-versus-NPZ wavelength differences.
#   - Verify the 218 original band IDs in both files.
#   - Permit only small metadata rounding differences (<= 0.1 nm).
#   - Use saved Step30 spectral-response WEIGHTS without modification.
#
# Original registration algorithms/parameters remain frozen.
# No external parameter tuning and no fusion-model inference.
#
# Outputs saved only to a NEW Step41C folder.
# =============================================================================

from google.colab import drive
from pathlib import Path
from scipy.ndimage import gaussian_filter, sobel

import ast
import hashlib
import json
import time

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

NOTEBOOK = (
    ROOT / "Colab Notebooks"
    / "Sensitivity of Hyperspectral and Multispectral Image .ipynb"
)

S13 = PROJECT / "Step13_Identity_Preserving_Gate"
S20 = PROJECT / "Step20_Bounded_Phase_Correlation"
S30 = PROJECT / "Step30_Frozen_Potsdam_Benchmark"
S39 = PROJECT / "Step39_Original_Estimator_Numerical_Replay"
S40 = PROJECT / "Step40_Reproduction_Discrepancy_Audit"

OUT = PROJECT / "Step41C_Frozen_Potsdam_Transfer"
OUT.mkdir(parents=True, exist_ok=True)

OUTPUT_NAMES = [
    "Step41C_Wavelength_Audit.csv",
    "Step41C_Registration_Results.csv",
    "Step41C_Method_Summary.csv",
    "Step41C_Shift_Summary.csv",
    "Step41C_Patch_Summary.csv",
    "Step41C_Paired_Comparison.csv",
    "Step41C_Provenance.json",
    "Step41C_Final_Report.json",
    "Step41C_Registration_Comparison.png",
    "Step41C_Registration_Comparison.pdf",
]

if any((OUT / name).exists() for name in OUTPUT_NAMES):
    raise FileExistsError(
        "Previous Step41C output exists. "
        "Preserve it and use a new output folder."
    )

START = time.monotonic()

def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def text_sha(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def save_json(path, value):
    path.write_text(
        json.dumps(
            value,
            indent=2,
            allow_nan=False,
            default=str
        ),
        encoding="utf-8"
    )

print("=" * 92, flush=True)
print("STEP 41C — FROZEN ORIGINAL-METHOD POTSDAM TRANSFER", flush=True)
print("=" * 92, flush=True)


# =============================================================================
# A — VERIFY FROZEN INPUTS
# =============================================================================

FILES = {
    "notebook": NOTEBOOK,
    "gate": S13 / "Step13_Frozen_Gate.json",
    "phase": S20 / "Step20_Frozen_Method.json",
    "protocol": S30 / "Step30_Frozen_Forward_Model.json",
    "response": S30 / "Step30_Spectral_Response_Matrix.csv",
    "manifest": S30 / "Step30_Paired_Observation_Manifest.csv",
    "functions": S39 / "Step39_Function_Provenance.csv",
    "reproduction": S40 / "Step40_Final_Report.json",
}

for label, path in FILES.items():
    if not path.is_file():
        raise FileNotFoundError(f"{label}: {path}")

gate_cfg = read_json(FILES["gate"])
phase_cfg = read_json(FILES["phase"])
protocol = read_json(FILES["protocol"])
reproduction = read_json(FILES["reproduction"])

assert reproduction["status"] == "NUMERICAL_REPRODUCTION_CONSISTENT"
assert reproduction["ncc_matches_at_1e_minus_5"] == 680
assert reproduction["phase_matches_at_3e_minus_5"] == 680
assert reproduction["phase_identity_snap_decisions_changed"] == 0

EXPECTED_PROTOCOL_SHA = (
    "dddd0df0d1bd0aeca0976e5bd22821ac4d07c3b1e8ed96c64eb3db4969a94eca"
)

assert file_sha(FILES["protocol"]) == EXPECTED_PROTOCOL_SHA

assert protocol["source_scene_count"] == 1
assert protocol["retained_band_count"] == 218
assert len(protocol["selected_patch_ids"]) == 5

GATE_NCC = float(gate_cfg["threshold_pixels"])
GATE_PHASE = float(phase_cfg["tau"])

RADIUS_NCC = 5
RADIUS_PHASE = int(phase_cfg["radius"])

BIAS_NCC_DY = -1
BIAS_NCC_DX = -1

BIAS_PHASE_DY = float(phase_cfg["bias_dy"])
BIAS_PHASE_DX = float(phase_cfg["bias_dx"])

assert GATE_NCC == 0.225
assert GATE_PHASE == 0.25
assert RADIUS_PHASE == 5

print("\n41C-A — Historical provenance VERIFIED", flush=True)


# =============================================================================
# B — RECOVER EXACT ORIGINAL FUNCTION DEFINITIONS
# =============================================================================

nb = read_json(NOTEBOOK)

prior = pd.read_csv(FILES["functions"])

expected_hashes = {
    (int(r.cell_index), str(r.function_name)): str(r.source_sha256)
    for r in prior.itertuples(index=False)
}

NCC_NAMES = [
    "registration_features",
    "integer_shift_np",
    "ncc",
    "score_surface",
    "quadratic_vertex",
    "estimate_both",
]

PHASE_NAMES = [
    "standardized_gray",
    "feature_map",
    "phase_surface",
    "vertex",
    "select_bounded",
]

def source_cell(index):
    cell = nb["cells"][index]

    if cell.get("cell_type") != "code":
        raise RuntimeError(f"Cell {index} is not code")

    text = cell.get("source", "")

    return (
        "".join(text)
        if isinstance(text, list)
        else str(text)
    )

def original_node(index, function_name):
    source = source_cell(index)
    tree = ast.parse(source)
    lines = source.splitlines()

    matches = [
        n for n in tree.body
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
        and n.name == function_name
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Cannot uniquely recover {function_name} "
            f"from cell {index}"
        )

    node = matches[0]

    if node.decorator_list:
        raise RuntimeError(
            f"Unexpected decorators: {function_name}"
        )

    extracted = "\n".join(
        lines[node.lineno - 1:node.end_lineno]
    )

    actual = text_sha(extracted)
    expected = expected_hashes.get(
        (index, function_name)
    )

    if expected is None or actual != expected:
        raise RuntimeError(
            f"Historical source mismatch: {function_name}"
        )

    return node

def install_functions(nodes, namespace, label):
    module = ast.Module(
        body=nodes,
        type_ignores=[]
    )
    ast.fix_missing_locations(module)

    exec(
        compile(module, label, "exec"),
        namespace
    )

ncc_nodes = [
    original_node(15, name)
    for name in NCC_NAMES
]

phase_nodes = [
    original_node(24, name)
    for name in PHASE_NAMES
]

ncc_env = {
    "np": np,
    "gaussian_filter": gaussian_filter,
    "sobel": sobel,
    "RADIUS": RADIUS_NCC,
    "BIAS_DY": BIAS_NCC_DY,
    "BIAS_DX": BIAS_NCC_DX,
    "CROP_MARGIN": 8,
    "BLUR_SIGMA": 1.2,
    "INTENSITY_WEIGHT": 0.7,
    "GRADIENT_WEIGHT": 0.3,
}

phase_env = {
    "np": np,
    "cv2": cv2,
}

install_functions(
    ncc_nodes, ncc_env, "mdas_original_ncc"
)

install_functions(
    phase_nodes, phase_env, "mdas_original_phase"
)

print(
    "41C-B — 11 original function definitions verified",
    flush=True
)


# =============================================================================
# C — VERIFIED 72->96 PHASE SURFACE ADAPTATION
# =============================================================================

original_phase_surface = phase_env["phase_surface"]

def generalized_phase_surface(reference, observed):

    reference = np.asarray(reference)
    observed = np.asarray(observed)

    if (
        reference.ndim != 2
        or reference.shape != observed.shape
    ):
        raise ValueError(
            "Phase inputs must be matching 2-D arrays"
        )

    h, w = reference.shape

    if min(h, w) < 2 * RADIUS_PHASE + 3:
        raise ValueError(
            "Phase image dimensions are too small"
        )

    window = np.outer(
        np.hanning(h),
        np.hanning(w)
    ).astype(np.float64)

    a = reference.astype(np.float64)
    b = observed.astype(np.float64)

    a = (a - a.mean()) * window
    b = (b - b.mean()) * window

    fa = np.fft.fft2(a)
    fb = np.fft.fft2(b)

    cross = fb * np.conj(fa)
    magnitude = np.abs(cross)

    scale = max(
        float(magnitude.max()),
        1e-12
    )

    normalized = cross / np.maximum(
        magnitude,
        scale * 1e-8
    )

    correlation = np.fft.fftshift(
        np.fft.ifft2(normalized).real
    )

    if not np.isfinite(correlation).all():
        raise RuntimeError(
            "Nonfinite phase-correlation surface"
        )

    return correlation

rng = np.random.default_rng(4102)

surface_differences = []

for _ in range(5):
    a = rng.normal(size=(72, 72)).astype(np.float32)
    b = rng.normal(size=(72, 72)).astype(np.float32)

    original = original_phase_surface(a, b)
    adapted = generalized_phase_surface(a, b)

    surface_differences.append(
        float(np.max(np.abs(original - adapted)))
    )

max_surface_difference = max(surface_differences)

if max_surface_difference > 1e-12:
    raise RuntimeError(
        "Phase adaptation failed 72x72 equivalence"
    )

toy = rng.normal(
    size=(96, 96)
).astype(np.float32)

toy_shifted = np.roll(
    toy,
    shift=(2, -3),
    axis=(0, 1)
)

toy_peak = phase_env["select_bounded"](
    generalized_phase_surface(
        toy,
        toy_shifted
    ),
    RADIUS_PHASE
)

assert np.allclose(
    [
        toy_peak["raw_dy"],
        toy_peak["raw_dx"]
    ],
    [2, -3],
    atol=0.1,
    rtol=0
)

phase_env["phase_surface"] = generalized_phase_surface

print("\n41C-C — Spatial adaptation VERIFIED", flush=True)
print(
    "72x72 maximum difference:",
    max_surface_difference
)
print(
    "96x96 known-shift estimate:",
    toy_peak["raw_dy"],
    toy_peak["raw_dx"]
)


# =============================================================================
# D — LOAD FROZEN SPECTRAL RESPONSE WITHOUT ALTERATION
# =============================================================================

response_df = pd.read_csv(
    FILES["response"]
)

weight_columns = [
    "synthetic_blue_weight",
    "synthetic_green_weight",
    "synthetic_red_weight",
    "synthetic_nir_weight",
]

required_columns = [
    "original_hsi_band",
    "wavelength_nm",
] + weight_columns

if not set(required_columns).issubset(
    response_df.columns
):
    raise RuntimeError(
        "Unexpected Step30 response schema:\n"
        + str(response_df.columns.tolist())
    )

if len(response_df) != 218:
    raise RuntimeError(
        "Unexpected response band count"
    )

retained_ids = np.asarray(
    protocol["retained_band_ids_1_based"],
    dtype=int
)

csv_ids = response_df[
    "original_hsi_band"
].to_numpy(dtype=int)

if not np.array_equal(csv_ids, retained_ids):
    raise RuntimeError(
        "Response CSV original band IDs do not "
        "match the frozen protocol"
    )

csv_wavelengths = response_df[
    "wavelength_nm"
].to_numpy(dtype=np.float64)

if not np.isfinite(csv_wavelengths).all():
    raise RuntimeError(
        "Nonfinite CSV wavelengths"
    )

RESPONSE = response_df[
    weight_columns
].to_numpy(dtype=np.float64).T

assert RESPONSE.shape == (4, 218)
assert np.isfinite(RESPONSE).all()

if not np.allclose(
    RESPONSE.sum(axis=1),
    1.0,
    rtol=0,
    atol=1e-7
):
    raise RuntimeError(
        "Response weights are not normalized"
    )

print("\n41C-D — Frozen response loaded", flush=True)
print("Original bands:", len(retained_ids))
print("Response shape:", RESPONSE.shape)
print("Channel sums:", RESPONSE.sum(axis=1))


# =============================================================================
# E — PRE-FLIGHT ALL PATCHES AND AUDIT WAVELENGTH DIFFERENCES
# =============================================================================

WAVELENGTH_MAX_TOLERANCE_NM = 0.1

patch_files = {}
patch_hashes = {}
audit_rows = []

print("\n41C-E — WAVELENGTH PROVENANCE AUDIT", flush=True)

for patch_id in protocol["selected_patch_ids"]:

    path = (
        S30 / "Paired_Observations"
        / f"{patch_id}.npz"
    )

    if not path.is_file():
        raise FileNotFoundError(path)

    patch_files[patch_id] = path
    patch_hashes[patch_id] = file_sha(path)

    with np.load(
        path,
        allow_pickle=False
    ) as z:
        ids = z["retained_band_ids"].astype(int)
        wavelengths = z["wavelengths_nm"].astype(np.float64)

        shapes = {
            "lrhsi": z["lrhsi"].shape,
            "hrmsi": z["hrmsi"].shape,
            "shifted_msi": z["shifted_msi"].shape,
            "shift_offsets_dy_dx": z["shift_offsets_dy_dx"].shape,
        }

    assert ids.shape == (218,)
    assert wavelengths.shape == (218,)

    assert shapes["lrhsi"] == (218, 32, 32)
    assert shapes["hrmsi"] == (4, 96, 96)
    assert shapes["shifted_msi"] == (17, 4, 96, 96)
    assert shapes["shift_offsets_dy_dx"] == (17, 2)

    ids_match = np.array_equal(
        ids, retained_ids
    )

    if not np.isfinite(wavelengths).all():
        raise RuntimeError(
            f"Nonfinite wavelengths in {patch_id}"
        )

    differences = np.abs(
        wavelengths - csv_wavelengths
    )

    for i, difference in enumerate(differences):

        audit_rows.append({
            "patch_id": patch_id,
            "band_index_0based": i,
            "original_band_id": int(ids[i]),
            "band_id_matches_protocol": bool(
                ids[i] == retained_ids[i]
            ),
            "npz_wavelength_nm": float(
                wavelengths[i]
            ),
            "csv_wavelength_nm": float(
                csv_wavelengths[i]
            ),
            "absolute_difference_nm": float(
                difference
            )
        })

    max_diff = float(differences.max())
    median_diff = float(np.median(differences))

    print(
        f"  {patch_id} | "
        f"IDs match: {ids_match} | "
        f"max wavelength difference: {max_diff:.8f} nm | "
        f"median: {median_diff:.8f} nm",
        flush=True
    )

wavelength_audit = pd.DataFrame(
    audit_rows
)

wavelength_audit.to_csv(
    OUT / "Step41C_Wavelength_Audit.csv",
    index=False
)

max_wavelength_difference = float(
    wavelength_audit["absolute_difference_nm"].max()
)

all_ids_match = bool(
    wavelength_audit[
        "band_id_matches_protocol"
    ].all()
)

if not all_ids_match:
    raise RuntimeError(
        "Actual original spectral band IDs are inconsistent. "
        "Do not run registration until resolved."
    )

if max_wavelength_difference > WAVELENGTH_MAX_TOLERANCE_NM:

    worst = wavelength_audit.nlargest(
        10,
        "absolute_difference_nm"
    )

    print("\nLARGEST WAVELENGTH DISCREPANCIES")
    print(
        worst.to_string(index=False)
    )

    raise RuntimeError(
        "Wavelength differences exceed 0.1 nm. "
        "This cannot yet be classified as CSV rounding. "
        "Review the saved Step41C_Wavelength_Audit.csv."
    )

print("\nSpectral metadata audit PASSED.")
print(
    "Maximum absolute wavelength difference:",
    max_wavelength_difference,
    "nm"
)

# This audit accepts small metadata differences only.
# The response weights are never recomputed or modified.


# =============================================================================
# F — BUILD EXTERNAL REGISTRATION REFERENCE
# =============================================================================

def registration_reference(lrhsi):

    if lrhsi.shape != (218, 32, 32):
        raise ValueError(
            f"Unexpected LRHSI shape: {lrhsi.shape}"
        )

    lowres_msi = np.einsum(
        "bc,chw->bhw",
        RESPONSE,
        lrhsi.astype(np.float64),
        optimize=True
    ).astype(np.float32)

    x = torch.from_numpy(
        np.ascontiguousarray(
            lowres_msi[None]
        )
    )

    with torch.inference_mode():
        y = F.interpolate(
            x,
            size=(96, 96),
            mode="bicubic",
            align_corners=False
        )

    output = (
        y[0].permute(1, 2, 0)
        .cpu().numpy()
        .astype(np.float32)
    )

    assert output.shape == (96, 96, 4)
    assert np.isfinite(output).all()

    return output

def msi_hwc(msi):

    if msi.shape != (4, 96, 96):
        raise ValueError(
            f"Unexpected MSI shape: {msi.shape}"
        )

    result = np.ascontiguousarray(
        np.moveaxis(msi, 0, -1),
        dtype=np.float32
    )

    assert np.isfinite(result).all()

    return result


# =============================================================================
# G — ORIGINAL REGISTRATION WRAPPERS
# =============================================================================

def original_ncc(target_features, observed):

    out = ncc_env["estimate_both"](
        target_features,
        observed
    )

    dy = float(out["refined_dy"])
    dx = float(out["refined_dx"])

    snapped = bool(
        np.hypot(dy, dx) < GATE_NCC
    )

    return {
        "raw_dy": dy,
        "raw_dx": dx,
        "estimated_dy": 0.0 if snapped else dy,
        "estimated_dx": 0.0 if snapped else dx,
        "snapped": snapped
    }

def original_phase(target_features, observed):

    obs_features = phase_env["feature_map"](
        observed,
        "phase_gradient"
    )

    surface = phase_env["phase_surface"](
        target_features,
        obs_features
    )

    out = phase_env["select_bounded"](
        surface,
        RADIUS_PHASE
    )

    raw_dy = float(out["raw_dy"])
    raw_dx = float(out["raw_dx"])

    corrected_dy = raw_dy - BIAS_PHASE_DY
    corrected_dx = raw_dx - BIAS_PHASE_DX

    snapped = bool(
        np.hypot(
            corrected_dy,
            corrected_dx
        ) < GATE_PHASE
    )

    return {
        "raw_dy": raw_dy,
        "raw_dx": raw_dx,
        "estimated_dy": (
            0.0 if snapped else corrected_dy
        ),
        "estimated_dx": (
            0.0 if snapped else corrected_dx
        ),
        "snapped": snapped
    }

def record_result(
    patch_id,
    condition_index,
    method,
    true_dy,
    true_dx,
    estimate
):

    true_dy = float(true_dy)
    true_dx = float(true_dx)

    dy = float(
        estimate["estimated_dy"]
    )

    dx = float(
        estimate["estimated_dx"]
    )

    epe = float(
        np.hypot(
            dy - true_dy,
            dx - true_dx
        )
    )

    return {
        "patch_id": patch_id,
        "condition_index": int(condition_index),
        "method": method,
        "true_dy": true_dy,
        "true_dx": true_dx,
        "shift_magnitude": float(
            np.hypot(true_dy, true_dx)
        ),
        "raw_dy": float(estimate["raw_dy"]),
        "raw_dx": float(estimate["raw_dx"]),
        "estimated_dy": dy,
        "estimated_dx": dx,
        "snapped": bool(estimate["snapped"]),
        "error_dy": dy - true_dy,
        "error_dx": dx - true_dx,
        "epe": epe,
        "within_025": bool(epe <= 0.25),
        "within_050": bool(epe <= 0.50),
        "aligned_preserved": bool(
            true_dy == 0
            and true_dx == 0
            and dy == 0
            and dx == 0
        )
    }


# =============================================================================
# H — REGISTRATION EVALUATION
# =============================================================================

print("\n41C-F — POTSDAM REGISTRATION EVALUATION", flush=True)

rows = []

for patch_number, patch_id in enumerate(
    protocol["selected_patch_ids"], 1
):

    with np.load(
        patch_files[patch_id],
        allow_pickle=False
    ) as z:

        lrhsi = z["lrhsi"].copy()
        hrmsi = z["hrmsi"].copy()
        shifted = z["shifted_msi"].copy()
        offsets = z["shift_offsets_dy_dx"].copy()

    if not (
        np.isfinite(lrhsi).all()
        and np.isfinite(hrmsi).all()
        and np.isfinite(shifted).all()
    ):
        raise RuntimeError(
            f"Nonfinite input data: {patch_id}"
        )

    aligned_indices = np.flatnonzero(
        np.all(offsets == 0, axis=1)
    )

    if len(aligned_indices) != 1:
        raise RuntimeError(
            f"Expected one aligned condition: {patch_id}"
        )

    if not np.array_equal(
        shifted[int(aligned_indices[0])],
        hrmsi
    ):
        raise RuntimeError(
            f"Aligned MSI is inconsistent: {patch_id}"
        )

    reference = registration_reference(lrhsi)

    ncc_reference_features = (
        ncc_env["registration_features"](
            reference
        )
    )

    phase_reference_features = (
        phase_env["feature_map"](
            reference,
            "phase_gradient"
        )
    )

    for condition_index, (dy, dx) in enumerate(offsets):

        observed = msi_hwc(
            shifted[condition_index]
        )

        ncc_out = original_ncc(
            ncc_reference_features,
            observed
        )

        phase_out = original_phase(
            phase_reference_features,
            observed
        )

        identity_out = {
            "raw_dy": 0.0,
            "raw_dx": 0.0,
            "estimated_dy": 0.0,
            "estimated_dx": 0.0,
            "snapped": False
        }

        for method, estimate in [
            ("frozen_ncc_step13", ncc_out),
            (
                "frozen_phase_step20_size_adapted",
                phase_out
            ),
            ("identity", identity_out)
        ]:

            rows.append(
                record_result(
                    patch_id,
                    condition_index,
                    method,
                    dy,
                    dx,
                    estimate
                )
            )

    print(
        f"  {patch_number}/5 {patch_id}: "
        "17 conditions × 3 methods complete",
        flush=True
    )

results = pd.DataFrame(rows)

assert len(results) == 255
assert results["patch_id"].nunique() == 5
assert (results.groupby("method").size() == 85).all()

assert not results.duplicated(
    ["patch_id", "condition_index", "method"]
).any()

assert np.isfinite(
    results[["estimated_dy", "estimated_dx", "epe"]]
    .to_numpy(float)
).all()

results.to_csv(
    OUT / "Step41C_Registration_Results.csv",
    index=False
)


# =============================================================================
# I — PERFORMANCE SUMMARY
# =============================================================================

summary_rows = []

for method, group in results.groupby("method"):

    aligned = group[
        group["shift_magnitude"] == 0
    ]

    summary_rows.append({
        "method": method,
        "cases": len(group),
        "mean_epe": float(group["epe"].mean()),
        "median_epe": float(group["epe"].median()),
        "max_epe": float(group["epe"].max()),
        "within_025": float(group["within_025"].mean()),
        "within_050": float(group["within_050"].mean()),
        "aligned_preserved": int(
            aligned["aligned_preserved"].sum()
        ),
        "aligned_mean_epe": float(
            aligned["epe"].mean()
        ),
        "mean_error_dy": float(
            group["error_dy"].mean()
        ),
        "mean_error_dx": float(
            group["error_dx"].mean()
        )
    })

summary = pd.DataFrame(summary_rows)

summary.to_csv(
    OUT / "Step41C_Method_Summary.csv",
    index=False
)

shift_summary = (
    results.groupby(
        ["method", "shift_magnitude"],
        as_index=False
    )
    .agg(
        cases=("epe", "size"),
        mean_epe=("epe", "mean"),
        median_epe=("epe", "median"),
        max_epe=("epe", "max"),
        within_025=("within_025", "mean")
    )
)

shift_summary.to_csv(
    OUT / "Step41C_Shift_Summary.csv",
    index=False
)

patch_summary = (
    results.groupby(
        ["patch_id", "method"],
        as_index=False
    )
    .agg(
        cases=("epe", "size"),
        mean_epe=("epe", "mean"),
        max_epe=("epe", "max")
    )
)

patch_summary.to_csv(
    OUT / "Step41C_Patch_Summary.csv",
    index=False
)

ncc = results[
    results["method"] == "frozen_ncc_step13"
][["patch_id", "condition_index", "epe"]].rename(
    columns={"epe": "ncc_epe"}
)

phase = results[
    results["method"] == "frozen_phase_step20_size_adapted"
][["patch_id", "condition_index", "epe"]].rename(
    columns={"epe": "phase_epe"}
)

paired = ncc.merge(
    phase,
    on=["patch_id", "condition_index"],
    validate="one_to_one"
)

paired["phase_advantage_px"] = (
    paired["ncc_epe"] - paired["phase_epe"]
)

paired["better_method"] = np.select(
    [
        paired["ncc_epe"] < paired["phase_epe"],
        paired["phase_epe"] < paired["ncc_epe"]
    ],
    ["ncc", "phase"],
    default="tie"
)

paired.to_csv(
    OUT / "Step41C_Paired_Comparison.csv",
    index=False
)

print("\n" + "=" * 92)
print("41C-G — EXTERNAL REGISTRATION RESULTS")
print("=" * 92)

print(
    summary.round(6).to_string(index=False)
)

print("\nPaired method comparison:")
print(
    paired["better_method"]
    .value_counts()
    .to_string()
)

print(
    "\nMean phase advantage:",
    float(paired["phase_advantage_px"].mean()),
    "pixels"
)


# =============================================================================
# J — PUBLICATION-QUALITY FIGURE
# =============================================================================

styles = {
    "frozen_ncc_step13": (
        "Frozen NCC + gate", "#225F8D", "o"
    ),
    "frozen_phase_step20_size_adapted": (
        "Frozen bounded phase", "#D0782C", "s"
    ),
    "identity": (
        "No correction", "#6B6B6B", "^"
    )
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "pdf.fonttype": 42,
    "savefig.facecolor": "white",
})

fig, ax = plt.subplots(figsize=(8.2, 5.1))

for method, (label, color, marker) in styles.items():

    g = shift_summary[
        shift_summary["method"] == method
    ].sort_values("shift_magnitude")

    ax.plot(
        g["shift_magnitude"],
        g["mean_epe"],
        color=color,
        marker=marker,
        linewidth=1.9,
        markersize=6,
        label=label
    )

ax.set_xlabel(
    "Applied displacement magnitude (pixels)"
)

ax.set_ylabel(
    "Mean endpoint error (pixels)"
)

ax.set_xticks(
    sorted(
        shift_summary["shift_magnitude"].unique()
    )
)

ax.grid(
    axis="y",
    alpha=0.2,
    linewidth=0.7
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False,
    fontsize=9
)

fig.tight_layout()

fig.savefig(
    OUT / "Step41C_Registration_Comparison.png",
    dpi=900,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    OUT / "Step41C_Registration_Comparison.pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)


# =============================================================================
# K — PROVENANCE AND SCIENTIFIC STATUS
# =============================================================================

provenance = {
    "experiment": (
        "Frozen MDAS registration on Potsdam "
        "with documented spatial and spectral interfaces"
    ),
    "historical_notebook_sha256": file_sha(NOTEBOOK),
    "historical_source_function_count": 11,
    "historical_ncc_reproduction": "680/680 at 1e-5 pixels",
    "historical_phase_reproduction": "680/680 at 3e-5 pixels",
    "phase_72x72_max_surface_difference": (
        max_surface_difference
    ),
    "phase_96x96_known_translation": {
        "actual": [2.0, -3.0],
        "estimated": [
            float(toy_peak["raw_dy"]),
            float(toy_peak["raw_dx"])
        ]
    },
    "frozen_ncc_settings": {
        "radius": RADIUS_NCC,
        "bias_dy": BIAS_NCC_DY,
        "bias_dx": BIAS_NCC_DX,
        "intensity_weight": 0.7,
        "gradient_weight": 0.3,
        "crop_margin": 8,
        "blur_sigma": 1.2,
        "identity_gate": GATE_NCC
    },
    "frozen_phase_settings": {
        "radius": RADIUS_PHASE,
        "bias_dy": BIAS_PHASE_DY,
        "bias_dx": BIAS_PHASE_DX,
        "identity_gate": GATE_PHASE
    },
    "spectral_interface": (
        "Direct frozen Step30 spectral response weights "
        "from original band IDs to four synthetic MSI channels"
    ),
    "maximum_wavelength_metadata_difference_nm": (
        max_wavelength_difference
    ),
    "accepted_wavelength_metadata_tolerance_nm": (
        WAVELENGTH_MAX_TOLERANCE_NM
    ),
    "original_band_id_order_verified": True,
    "response_weights_modified": False,
    "registration_parameters_modified": False,
    "external_ground_truth_used_for_tuning": False,
    "source_scene_count": 1,
    "source_patch_count": 5,
    "conditions_per_patch": 17,
    "step30_protocol_sha256": file_sha(FILES["protocol"]),
    "step30_response_sha256": file_sha(FILES["response"]),
    "step40_report_sha256": file_sha(FILES["reproduction"]),
    "patch_npz_sha256": patch_hashes,
}

save_json(
    OUT / "Step41C_Provenance.json",
    provenance
)

report = {
    "status": "EXTERNAL_FROZEN_REGISTRATION_EVALUATION_COMPLETE",
    "source_scenes": 1,
    "source_patches": 5,
    "shift_conditions_per_patch": 17,
    "total_records": len(results),
    "method_summary": summary.to_dict("records"),
    "paired_comparison": {
        "ncc_better": int(
            (paired["better_method"] == "ncc").sum()
        ),
        "phase_better": int(
            (paired["better_method"] == "phase").sum()
        ),
        "ties": int(
            (paired["better_method"] == "tie").sum()
        ),
        "mean_phase_advantage_px": float(
            paired["phase_advantage_px"].mean()
        )
    },
    "max_wavelength_metadata_difference_nm": (
        max_wavelength_difference
    ),
    "registration_parameters_retained": True,
    "external_spectral_interface_adapted": True,
    "phase_spatial_dimensions_adapted": True,
    "external_fusion_checkpoint_tested": False,
    "limitations": [
        "Only one independent Potsdam acquisition.",
        "Five patches are not independent source scenes.",
        "MSI and translations are synthetic.",
        "Original MDAS spectral ridge projection cannot "
        "directly accept the 218-band EnMAP input.",
        "The frozen Step30 response defines the external "
        "four-channel sensor interface.",
        "The original MDAS validation-derived biases "
        "are deliberately retained.",
        "The 72x72 phase surface requires a documented "
        "96x96 dimensional generalization.",
        "No external fusion model performance is evaluated."
    ],
    "elapsed_seconds": round(
        time.monotonic() - START, 2
    ),
    "output_directory": str(OUT)
}

save_json(
    OUT / "Step41C_Final_Report.json",
    report
)

print("\n" + "=" * 92)
print("STEP 41C — FINAL REPORT")
print("=" * 92)
print(json.dumps(report, indent=2))

print("\nSTEP 41C COMPLETE")
print("Historical algorithms and all previous outputs preserved.")


In [ ]:

# STEP 41D — VERIFY EXACT POTSDAM RESAMPLING IMPLEMENTATION

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json
import re

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

NOTEBOOK = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Sensitivity of Hyperspectral and Multispectral Image .ipynb"
)

if not NOTEBOOK.is_file():
    raise FileNotFoundError(
        f"Original execution notebook not found: {NOTEBOOK}"
    )

nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))

OUT = PROJECT / (
    "Step41D_Potsdam_Interpolation_Audit_"
    + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
)
OUT.mkdir(parents=True, exist_ok=False)

markers = [
    "STEP 41C",
    "STEP 41B",
    "FROZEN ORIGINAL-METHOD POTSDAM TRANSFER",
    "FROZEN POTSDAM TRANSFER",
]

operators = [
    r"\binterpolate\s*\(",
    r"\bresize\s*\(",
    r"\bzoom\s*\(",
    r"\bmap_coordinates\s*\(",
    r"\bRegularGridInterpolator\s*\(",
    r"\bgrid_sample\s*\(",
    r"\bupsample\b",
    r"\bINTER_LINEAR\b",
    r"\bINTER_CUBIC\b",
    r"\bbilinear\b",
    r"\bbicubic\b",
    r"\border\s*=",
]

print("=" * 80)
print("STEP 41D — POTSDAM INTERPOLATION SOURCE AUDIT")
print("=" * 80)

candidate_cells = []

for index, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))

    if any(
        marker.lower() in src.lower()
        for marker in markers
    ):
        candidate_cells.append((index, src))

print("Candidate source cells:", [i for i, _ in candidate_cells])

if not candidate_cells:
    print(
        "No explicit Step 41B/41C marker found. "
        "The current notebook may not contain "
        "the original execution cell."
    )

report = {
    "notebook": str(NOTEBOOK),
    "candidate_cells": [],
    "interpolation_verified": False,
    "conclusion": (
        "Manual source review required; no interpolation "
        "mode inferred from the numerical results."
    )
}

for index, src in candidate_cells:

    lines = src.splitlines()
    matches = []

    print("\n" + "=" * 80)
    print(f"CELL {index} — {len(lines)} source lines")
    print("=" * 80)

    for n, line in enumerate(lines):

        if any(
            re.search(pattern, line, re.IGNORECASE)
            for pattern in operators
        ):

            start = max(0, n - 5)
            end = min(len(lines), n + 7)

            context = "\n".join(
                f"{j+1:04d}: {lines[j]}"
                for j in range(start, end)
            )

            matches.append({
                "line": n + 1,
                "context": context
            })

            print("\n" + context)

    report["candidate_cells"].append({
        "cell_index": index,
        "match_count": len(matches),
        "matches": matches
    })

    (OUT / f"Step41D_Cell_{index}_Source.py").write_text(
        src,
        encoding="utf-8"
    )

(OUT / "Step41D_Audit.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 80)
print("STEP 41D AUDIT COMPLETE")
print("=" * 80)
print("Output:", OUT)
print(
    "Review the printed resampling calls and identify "
    "the one constructing the low-resolution-HSI-derived "
    "registration reference."
)


In [ ]:

# =============================================================================
# STEP 41E — POTSDAM REGISTRATION INTERPOLATION SOURCE RECOVERY
# =============================================================================
# Searches all saved Colab notebooks and MDAS project Python scripts.
# Does not execute historical code, rerun estimators, or modify old results.
# =============================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import hashlib
import time
import os

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

COLAB_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks"
)

RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_%f"
)

OUT = PROJECT / (
    "Step41E_Interpolation_Source_Recovery_" + RUN_ID
)

OUT.mkdir(parents=True, exist_ok=False)

START = time.monotonic()

print("=" * 88)
print("STEP 41E — POTSDAM INTERPOLATION SOURCE RECOVERY")
print("=" * 88)

# ----------------------------------------------------------------------
# A. Search phrases linked to the actual Step 41C program.
# ----------------------------------------------------------------------

STRONG_MARKERS = [
    "STEP 41C",
    "STEP 41B",
    "original_phase_transfer",
    "original_ncc_transfer",
    "phase_target_features",
    "target_phase_feature",
    "Wavelength mismatch for",
    "EXTERNAL_FROZEN_REGISTRATION_EVALUATION_COMPLETE",
    "frozen_phase_step20_size_adapted",
    "frozen_ncc_step13",
    "phase_96x96_known_translation",
]

SECONDARY_MARKERS = [
    "potsdam_r0_c192",
    "potsdam_r96_c96",
    "Step30_Frozen_Potsdam_Benchmark",
    "Step41C_Frozen_Potsdam_Transfer",
    "phase_surface",
    "select_bounded",
]

RESAMPLING_PATTERNS = {
    "torch_interpolate": r"\b(?:F|torch\.nn\.functional)\.interpolate\s*\(",
    "opencv_resize": r"\bcv2\.resize\s*\(",
    "scipy_zoom": r"\b(?:ndimage\.)?zoom\s*\(",
    "skimage_resize": r"\b(?:transform\.)?resize\s*\(",
    "opencv_interpolation": r"\bcv2\.INTER_(?:LINEAR|CUBIC|AREA|NEAREST)\b",
    "bilinear": r"\bbilinear\b",
    "bicubic": r"\bbicubic\b",
    "scipy_order": r"\border\s*=\s*[0-5]\b",
    "grid_sample": r"\bgrid_sample\s*\(",
    "repeat_resampling": r"\bnp\.repeat\s*\(",
    "other_upsampling": r"\b(?:upsample|upsampling)\b",
}

# ----------------------------------------------------------------------
# B. Locate source files.
# ----------------------------------------------------------------------

ROOTS = [
    COLAB_DIR,
    PROJECT,
]

SKIP_DIRS = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    "node_modules",
    ".venv",
    "venv",
}

# Reading notebooks larger than 40 MB is unnecessary for this audit.
MAX_FILE_BYTES = 40 * 1024 * 1024

files = []
seen = set()

for root in ROOTS:

    if not root.is_dir():
        print("Missing search directory:", root)
        continue

    for dirpath, dirs, filenames in os.walk(root):

        dirs[:] = sorted(
            d for d in dirs
            if d not in SKIP_DIRS
        )

        for name in sorted(filenames):

            if not name.lower().endswith(
                (".ipynb", ".py")
            ):
                continue

            path = Path(dirpath) / name

            if str(path) in seen:
                continue

            seen.add(str(path))

            try:
                size = path.stat().st_size
            except OSError:
                continue

            if size > MAX_FILE_BYTES:
                continue

            files.append(path)

print("\nA — DISCOVERY")
print("Notebook and Python files:", len(files))

# ----------------------------------------------------------------------
# C. Extract code cells from notebooks.
# ----------------------------------------------------------------------

def get_units(path):

    if path.suffix.lower() == ".py":

        source = path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        return [{
            "cell": "python_file",
            "source": source
        }]

    notebook = json.loads(
        path.read_text(
            encoding="utf-8",
            errors="replace"
        )
    )

    units = []

    for index, cell in enumerate(
        notebook.get("cells", [])
    ):

        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", "")

        if isinstance(source, list):
            source = "".join(source)

        units.append({
            "cell": index,
            "source": source
        })

    return units


def hash_text(source):

    return hashlib.sha256(
        source.encode("utf-8")
    ).hexdigest()


def source_context(lines, index, before=7, after=12):

    lo = max(0, index - before)
    hi = min(len(lines), index + after + 1)

    return "\n".join(
        f"{k+1:05d}: {lines[k]}"
        for k in range(lo, hi)
    )


# ----------------------------------------------------------------------
# D. Identify source candidates and interpolation calls.
# ----------------------------------------------------------------------

candidates = []
errors = []

for file_number, path in enumerate(files, 1):

    try:
        units = get_units(path)

    except Exception as exc:

        errors.append({
            "path": str(path),
            "error": str(exc)
        })

        continue

    for unit in units:

        source = unit["source"]

        lower = source.lower()

        strong = [
            item
            for item in STRONG_MARKERS
            if item.lower() in lower
        ]

        secondary = [
            item
            for item in SECONDARY_MARKERS
            if item.lower() in lower
        ]

        # Do not mistake a generic resize script for Step 41C.
        if not strong and len(secondary) < 3:
            continue

        lines = source.splitlines()

        interpolation_calls = []

        for index, line in enumerate(lines):

            matches = [
                label
                for label, pattern in RESAMPLING_PATTERNS.items()
                if re.search(
                    pattern,
                    line,
                    flags=re.IGNORECASE
                )
            ]

            if matches:

                interpolation_calls.append({
                    "line": index + 1,
                    "type": matches,
                    "context": source_context(
                        lines,
                        index
                    )
                })

        # Higher score for distinctive Step 41C functions.
        score = (
            10 * len(strong)
            + len(secondary)
            + min(len(interpolation_calls), 20)
        )

        candidates.append({
            "path": str(path),
            "cell": unit["cell"],
            "score": score,
            "strong_markers": strong,
            "secondary_markers": secondary,
            "source_sha256": hash_text(source),
            "source": source,
            "interpolation_calls": interpolation_calls,
        })

    if file_number % 50 == 0:

        print(
            f"Scanned {file_number}/{len(files)} files; "
            f"candidate units: {len(candidates)}"
        )

candidates.sort(
    key=lambda x: x["score"],
    reverse=True
)

print("\nB — SOURCE CANDIDATES")
print("Candidate code units found:", len(candidates))

# ----------------------------------------------------------------------
# E. Save source excerpts and print most relevant calls.
# ----------------------------------------------------------------------

manifest = []

TEXT_REPORT = OUT / "Step41E_Source_Inspection.txt"

with TEXT_REPORT.open(
    "w",
    encoding="utf-8"
) as report:

    for rank, item in enumerate(candidates, 1):

        record = {
            "rank": rank,
            "file": item["path"],
            "cell": item["cell"],
            "score": item["score"],
            "strong_markers": item["strong_markers"],
            "secondary_markers": item["secondary_markers"],
            "source_sha256": item["source_sha256"],
            "interpolation_call_count": len(
                item["interpolation_calls"]
            ),
        }

        manifest.append(record)

        # Retain the complete matching code unit,
        # including definitions that may be below a match.
        if rank <= 30:

            extracted = (
                OUT /
                f"Candidate_{rank:02d}_"
                f"Cell_{item['cell']}.py"
            )

            extracted.write_text(
                item["source"],
                encoding="utf-8"
            )

        report.write("\n" + "=" * 88 + "\n")
        report.write(f"CANDIDATE {rank}\n")
        report.write("=" * 88 + "\n")

        report.write(
            "File: " + item["path"] + "\n"
        )

        report.write(
            "Cell: " + str(item["cell"]) + "\n"
        )

        report.write(
            "Score: " + str(item["score"]) + "\n"
        )

        report.write(
            "Strong markers: "
            + repr(item["strong_markers"])
            + "\n"
        )

        report.write(
            "Source SHA256: "
            + item["source_sha256"]
            + "\n"
        )

        report.write(
            "Interpolation-related matches: "
            + str(len(item["interpolation_calls"]))
            + "\n"
        )

        for call in item["interpolation_calls"]:

            report.write(
                "\nCALL at line "
                + str(call["line"])
                + "\n"
            )

            report.write(
                call["context"] + "\n"
            )


    report.write(
        "\nFILES THAT COULD NOT BE READ\n"
    )

    for entry in errors:

        report.write(
            repr(entry) + "\n"
        )

# Print the best candidates.
for rank, item in enumerate(candidates[:8], 1):

    print("\n" + "=" * 75)
    print(f"CANDIDATE {rank}")
    print("=" * 75)

    print("File:", item["path"])
    print("Cell:", item["cell"])
    print("Score:", item["score"])
    print("Markers:", item["strong_markers"])

    print(
        "Interpolation-related calls:",
        len(item["interpolation_calls"])
    )

    for call in item["interpolation_calls"][:12]:

        print(
            "\nLine",
            call["line"],
            "—",
            ", ".join(call["type"])
        )

        print(call["context"])

# ----------------------------------------------------------------------
# F. Save machine-readable audit.
# ----------------------------------------------------------------------

final_report = {
    "status": (
        "CANDIDATE_SOURCE_FOUND_REQUIRES_CODE_REVIEW"
        if candidates
        else "ORIGINAL_EXECUTION_SOURCE_NOT_LOCATED"
    ),
    "searched_roots": [str(x) for x in ROOTS],
    "files_scanned": len(files),
    "candidate_code_units": len(candidates),
    "candidate_inventory": manifest,
    "unreadable_files": errors,
    "interpolation_mode_verified": False,
    "original_algorithms_executed": False,
    "historical_outputs_modified": False,
    "elapsed_seconds": round(
        time.monotonic() - START, 2
    ),
    "output_directory": str(OUT)
}

(OUT / "Step41E_Final_Report.json").write_text(
    json.dumps(final_report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 88)
print("STEP 41E — FINAL STATUS")
print("=" * 88)

print("Status:", final_report["status"])
print("Files scanned:", len(files))
print("Candidate code units:", len(candidates))
print("Report:", TEXT_REPORT)
print("Output directory:", OUT)

print("\nIMPORTANT:")
print(
    "A detected resize or interpolate call is not automatically "
    "the Potsdam registration-reference upsampling operation."
)
print(
    "The relevant function and its input/output array shapes "
    "must be confirmed before the manuscript states an "
    "interpolation method."
)


In [ ]:

# ================================================================
# STEP 41F — CHECK ORIGINAL COLAB EXECUTION HISTORY
# Read-only inspection. No registration reruns or data changes.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import re

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

OUT = PROJECT / (
    "Step41F_Execution_History_Audit_"
    + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
)
OUT.mkdir(parents=True, exist_ok=False)

ip = get_ipython()

if ip is None:
    raise RuntimeError("This must run inside IPython/Colab.")

history = list(ip.user_ns.get("In", []))

markers = [
    "STEP 41C",
    "STEP 41B",
    "original_phase_transfer",
    "original_ncc_transfer",
    "phase_target_features",
    "frozen_phase_step20_size_adapted",
    "Wavelength mismatch for",
    "Step41C_Frozen_Potsdam_Transfer",
]

patterns = [
    r"\binterpolate\s*\(",
    r"\bresize\s*\(",
    r"\bzoom\s*\(",
    r"\bmap_coordinates\s*\(",
    r"\bgrid_sample\s*\(",
    r"\bINTER_(LINEAR|CUBIC|AREA|NEAREST)\b",
    r"\border\s*=\s*[0-5]\b",
    r"\bbilinear\b",
    r"\bbicubic\b",
    r"\bupsampl",
]

matches = []

print("=" * 78)
print("STEP 41F — COLAB EXECUTION HISTORY AUDIT")
print("=" * 78)
print("History entries:", len(history))

for cell_no, source in enumerate(history):

    if not isinstance(source, str):
        continue

    found = [
        marker for marker in markers
        if marker.lower() in source.lower()
    ]

    if not found:
        continue

    lines = source.splitlines()
    calls = []

    for line_no, line in enumerate(lines):

        if any(
            re.search(p, line, re.IGNORECASE)
            for p in patterns
        ):
            lo = max(0, line_no - 7)
            hi = min(len(lines), line_no + 9)

            context = "\n".join(
                f"{k+1:04d}: {lines[k]}"
                for k in range(lo, hi)
            )

            calls.append({
                "line": line_no + 1,
                "context": context
            })

    matches.append({
        "history_cell": cell_no,
        "markers": found,
        "calls": calls
    })

    print("\n" + "-" * 78)
    print("History cell:", cell_no)
    print("Step 41 markers:", found)
    print("Resampling-related matches:", len(calls))

    for call in calls[:15]:
        print("\n" + call["context"])

    # Save original matching source for manual verification.
    (OUT / f"History_Cell_{cell_no}.py").write_text(
        source,
        encoding="utf-8"
    )

report = {
    "history_entries": len(history),
    "matching_cells": len(matches),
    "matches": matches,
    "interpolation_verified": False,
    "note": (
        "A call must be traced to the 32x32-to-96x96 "
        "registration-reference construction before "
        "its interpolation mode can be verified."
    )
}

(OUT / "Step41F_Report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 78)
print("Matching execution cells:", len(matches))
print("Audit saved:", OUT)
print("=" * 78)


In [ ]:

# =============================================================================
# STEP 42 — EXTERNAL CALIBRATION TRANSFER FAILURE AUDIT
#
# Research objective:
#   Attribute the approximately (+1,+1) pixel external
#   residual to the original MDAS bias correction,
#   while distinguishing that effect from the underlying
#   registration estimator.
#
# Inputs:
#   Step41C_Registration_Results.csv
#   Step41C_Final_Report.json
#   Step13_Frozen_Gate.json
#   Step20_Frozen_Method.json
#
# Methods:
#   A. Frozen original NCC + gate (unchanged)
#   B. Frozen original bounded phase (unchanged)
#   C. No correction baseline
#   D. NCC pre-bias estimate + ORIGINAL identity gate
#   E. Phase pre-bias estimate + ORIGINAL identity gate
#
# D and E are POST HOC DIAGNOSTIC COUNTERFACTUALS,
# not externally validated competing methods.
#
# No registration functions rerun.
# No model parameters optimized.
# No historical or external files overwritten.
#
# Outputs:
#   Step42_Bias_Attribution_Cases.csv
#   Step42_Descriptive_Method_Summary.csv
#   Step42_Aligned_Residuals.csv
#   Step42_Patch_Level_Comparison.csv
#   Step42_Shift_Magnitude_Comparison.csv
#   Step42_Frozen_vs_Counterfactual.png/.pdf
#   Step42_Aligned_Residual_Vectors.png/.pdf
#   Step42_Final_Report.json
# =============================================================================

from google.colab import drive
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

drive.mount("/content/drive")

PROJECT = Path(
    "/content/drive/MyDrive/MDAS_Registration_Aware_Revision_2026"
)

S13 = PROJECT / "Step13_Identity_Preserving_Gate"
S20 = PROJECT / "Step20_Bounded_Phase_Correlation"
S41 = PROJECT / "Step41C_Frozen_Potsdam_Transfer"

OUT = PROJECT / "Step42_External_Calibration_Transfer_Audit"
OUT.mkdir(parents=True, exist_ok=True)

INPUT_RESULTS = S41 / "Step41C_Registration_Results.csv"
INPUT_REPORT = S41 / "Step41C_Final_Report.json"
NCC_CONFIG = S13 / "Step13_Frozen_Gate.json"
PHASE_CONFIG = S20 / "Step20_Frozen_Method.json"

OUTPUT_NAMES = [
    "Step42_Bias_Attribution_Cases.csv",
    "Step42_Descriptive_Method_Summary.csv",
    "Step42_Aligned_Residuals.csv",
    "Step42_Patch_Level_Comparison.csv",
    "Step42_Shift_Magnitude_Comparison.csv",
    "Step42_Frozen_vs_Counterfactual.png",
    "Step42_Frozen_vs_Counterfactual.pdf",
    "Step42_Aligned_Residual_Vectors.png",
    "Step42_Aligned_Residual_Vectors.pdf",
    "Step42_Final_Report.json",
]

existing = [
    x for x in OUTPUT_NAMES if (OUT / x).exists()
]
if existing:
    raise FileExistsError(
        "Step42 outputs already exist. Preserve them "
        "and use a different output directory."
    )

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def save_json(path, value):
    path.write_text(
        json.dumps(
            value, indent=2, allow_nan=False, default=str
        ),
        encoding="utf-8"
    )

print("=" * 88)
print("STEP 42 — EXTERNAL CALIBRATION TRANSFER FAILURE AUDIT")
print("=" * 88)

for path in [
    INPUT_RESULTS, INPUT_REPORT, NCC_CONFIG, PHASE_CONFIG
]:
    if not path.is_file():
        raise FileNotFoundError(path)

df = pd.read_csv(INPUT_RESULTS)
previous = load_json(INPUT_REPORT)
ncfg = load_json(NCC_CONFIG)
pcfg = load_json(PHASE_CONFIG)

assert previous["status"] == (
    "EXTERNAL_FROZEN_REGISTRATION_EVALUATION_COMPLETE"
)
assert len(df) == 255

NCC_METHOD = "frozen_ncc_step13"
PHASE_METHOD = "frozen_phase_step20_size_adapted"
IDENTITY = "identity"

assert set(df["method"]) == {
    NCC_METHOD, PHASE_METHOD, IDENTITY
}

KEYS = ["patch_id", "condition_index"]

assert not df.duplicated(KEYS + ["method"]).any()
assert df["patch_id"].nunique() == 5

for method in [NCC_METHOD, PHASE_METHOD, IDENTITY]:
    assert int((df["method"] == method).sum()) == 85

NCC_GATE = float(ncfg["threshold_pixels"])
PHASE_GATE = float(pcfg["tau"])

NCC_BIAS_DY = -1.0
NCC_BIAS_DX = -1.0

PHASE_BIAS_DY = float(pcfg["bias_dy"])
PHASE_BIAS_DX = float(pcfg["bias_dx"])

assert NCC_GATE == 0.225
assert PHASE_GATE == 0.25
assert int(pcfg["radius"]) == 5

numeric_cols = [
    "true_dy", "true_dx",
    "raw_dy", "raw_dx",
    "estimated_dy", "estimated_dx", "epe"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="raise")

assert np.isfinite(df[numeric_cols].to_numpy(float)).all()

print("\n42A — VERIFIED FROZEN INPUTS")
print("Records:", len(df))
print("Source patches:", df["patch_id"].nunique())
print("NCC historical bias:", NCC_BIAS_DY, NCC_BIAS_DX)
print("Phase historical bias:", PHASE_BIAS_DY, PHASE_BIAS_DX)

# -------------------------------------------------------------------------
# B — RECONSTRUCT PRE-BIAS DISPLACEMENTS
#
# Original NCC saved 'raw_dy/raw_dx' in Step41C
# actually contain the refined, BIAS-CORRECTED
# estimator outputs from estimate_both().
#
# Therefore:
#   NCC pre-bias = saved refined + historical bias
#
# Original phase saved 'raw_dy/raw_dx' are the genuine
# raw FFT displacements, before the bias correction.
# -------------------------------------------------------------------------

counterfactual_rows = []

for method, sub in df.groupby("method", sort=False):

    sub = sub.copy()

    if method == NCC_METHOD:

        pre_y = sub["raw_dy"].to_numpy(float) + NCC_BIAS_DY
        pre_x = sub["raw_dx"].to_numpy(float) + NCC_BIAS_DX

        gate = NCC_GATE
        name = "ncc_prebias_same_gate_diagnostic"

    elif method == PHASE_METHOD:

        pre_y = sub["raw_dy"].to_numpy(float)
        pre_x = sub["raw_dx"].to_numpy(float)

        gate = PHASE_GATE
        name = "phase_prebias_same_gate_diagnostic"

    else:
        continue

    mag = np.hypot(pre_y, pre_x)
    snap = mag < gate

    diagnostic_y = np.where(snap, 0.0, pre_y)
    diagnostic_x = np.where(snap, 0.0, pre_x)

    true_y = sub["true_dy"].to_numpy(float)
    true_x = sub["true_dx"].to_numpy(float)

    diagnostic_epe = np.hypot(
        diagnostic_y - true_y,
        diagnostic_x - true_x
    )

    for i, row in enumerate(sub.itertuples(index=False)):
        counterfactual_rows.append({
            "patch_id": row.patch_id,
            "condition_index": int(row.condition_index),
            "true_dy": float(true_y[i]),
            "true_dx": float(true_x[i]),
            "shift_magnitude": float(np.hypot(
                true_y[i], true_x[i]
            )),
            "method": name,
            "estimated_dy": float(diagnostic_y[i]),
            "estimated_dx": float(diagnostic_x[i]),
            "epe": float(diagnostic_epe[i]),
            "snapped": bool(snap[i]),
            "is_diagnostic_counterfactual": True,
            "prebias_dy": float(pre_y[i]),
            "prebias_dx": float(pre_x[i])
        })

diagnostics = pd.DataFrame(counterfactual_rows)

# Historical frozen results are retained directly from Step41C.
frozen = df[
    KEYS + [
        "true_dy", "true_dx", "method",
        "estimated_dy", "estimated_dx", "epe"
    ]
].copy()

frozen["shift_magnitude"] = np.hypot(
    frozen["true_dy"], frozen["true_dx"]
)
frozen["is_diagnostic_counterfactual"] = False
frozen["snapped"] = (
    df["snapped"].astype(str).str.lower().isin(["true", "1"])
)
frozen["prebias_dy"] = np.nan
frozen["prebias_dx"] = np.nan

all_cases = pd.concat(
    [frozen, diagnostics],
    ignore_index=True
)

assert len(all_cases) == 425
assert (all_cases.groupby("method").size() == 85).all()
assert not all_cases.duplicated(KEYS + ["method"]).any()

all_cases.to_csv(
    OUT / "Step42_Bias_Attribution_Cases.csv",
    index=False
)

# -------------------------------------------------------------------------
# C — VERIFY THAT FROZEN ORIGINAL RESULTS WERE NOT ALTERED
# -------------------------------------------------------------------------

for method in [NCC_METHOD, PHASE_METHOD, IDENTITY]:

    a = df[df["method"] == method].sort_values(KEYS)
    b = all_cases[
        all_cases["method"] == method
    ].sort_values(KEYS)

    assert np.array_equal(
        a["epe"].to_numpy(float),
        b["epe"].to_numpy(float)
    )

    assert np.array_equal(
        a["estimated_dy"].to_numpy(float),
        b["estimated_dy"].to_numpy(float)
    )

    assert np.array_equal(
        a["estimated_dx"].to_numpy(float),
        b["estimated_dx"].to_numpy(float)
    )

print("\n42B — ORIGINAL STEP41C RESULTS VERIFIED UNCHANGED")

# -------------------------------------------------------------------------
# D — METHOD SUMMARIES
# -------------------------------------------------------------------------

summary_rows = []

for method, group in all_cases.groupby("method"):

    aligned = group[
        group["shift_magnitude"] == 0.0
    ]

    summary_rows.append({
        "method": method,
        "status": (
            "POST_HOC_DIAGNOSTIC"
            if group["is_diagnostic_counterfactual"].iloc[0]
            else "FROZEN_ORIGINAL_OR_BASELINE"
        ),
        "cases": len(group),
        "mean_epe": float(group["epe"].mean()),
        "median_epe": float(group["epe"].median()),
        "max_epe": float(group["epe"].max()),
        "within_025": float((group["epe"] <= 0.25).mean()),
        "within_050": float((group["epe"] <= 0.50).mean()),
        "aligned_cases": len(aligned),
        "aligned_preserved": int((
            (aligned["estimated_dy"] == 0)
            & (aligned["estimated_dx"] == 0)
        ).sum()),
        "aligned_mean_epe": float(aligned["epe"].mean()),
        "signed_dy_bias": float(
            (group["estimated_dy"] - group["true_dy"]).mean()
        ),
        "signed_dx_bias": float(
            (group["estimated_dx"] - group["true_dx"]).mean()
        )
    })

summary = pd.DataFrame(summary_rows)

summary.to_csv(
    OUT / "Step42_Descriptive_Method_Summary.csv",
    index=False
)

print("\n42C — FROZEN VERSUS POST HOC DIAGNOSTIC")
print(summary.round(6).to_string(index=False))

# -------------------------------------------------------------------------
# E — PAIRED ATTRIBUTION AT THE SAME 85 CONDITIONS
# -------------------------------------------------------------------------

paired_rows = []

pairs = [
    (NCC_METHOD, "ncc_prebias_same_gate_diagnostic"),
    (
        PHASE_METHOD,
        "phase_prebias_same_gate_diagnostic"
    )
]

for frozen_name, diagnostic_name in pairs:

    f = all_cases[
        all_cases["method"] == frozen_name
    ][KEYS + ["epe", "estimated_dy", "estimated_dx"]].copy()

    d = all_cases[
        all_cases["method"] == diagnostic_name
    ][KEYS + ["epe", "estimated_dy", "estimated_dx"]].copy()

    combined = f.merge(
        d, on=KEYS,
        suffixes=("_frozen", "_prebias"),
        validate="one_to_one"
    )

    for row in combined.itertuples(index=False):
        paired_rows.append({
            "patch_id": row.patch_id,
            "condition_index": int(row.condition_index),
            "method_family": frozen_name,
            "frozen_epe": float(row.epe_frozen),
            "prebias_epe": float(row.epe_prebias),
            "diagnostic_epe_reduction": float(
                row.epe_frozen - row.epe_prebias
            )
        })

attribution = pd.DataFrame(paired_rows)

print("\n42D — POST HOC BIAS ATTRIBUTION")

for family, group in attribution.groupby("method_family"):
    print(
        family,
        "| frozen mean:",
        round(float(group["frozen_epe"].mean()), 6),
        "| prebias diagnostic mean:",
        round(float(group["prebias_epe"].mean()), 6),
        "| mean difference:",
        round(float(
            group["diagnostic_epe_reduction"].mean()
        ), 6)
    )

# -------------------------------------------------------------------------
# F — PATCH AND SHIFT GROUP SUMMARY
# -------------------------------------------------------------------------

patch_summary = (
    all_cases.groupby(
        ["patch_id", "method"],
        as_index=False
    )
    .agg(
        cases=("epe", "size"),
        mean_epe=("epe", "mean"),
        max_epe=("epe", "max")
    )
)

patch_summary.to_csv(
    OUT / "Step42_Patch_Level_Comparison.csv",
    index=False
)

shift_group = (
    all_cases.groupby(
        ["method", "shift_magnitude"],
        as_index=False
    )
    .agg(
        cases=("epe", "size"),
        mean_epe=("epe", "mean"),
        median_epe=("epe", "median"),
        max_epe=("epe", "max")
    )
)

shift_group.to_csv(
    OUT / "Step42_Shift_Magnitude_Comparison.csv",
    index=False
)

aligned = all_cases[
    all_cases["shift_magnitude"] == 0.0
].copy()

aligned["signed_error_dy"] = aligned["estimated_dy"]
aligned["signed_error_dx"] = aligned["estimated_dx"]

aligned.to_csv(
    OUT / "Step42_Aligned_Residuals.csv",
    index=False
)

# -------------------------------------------------------------------------
# G — FIGURE 1: MEAN EPE BY DISPLACEMENT
# -------------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
})

styles = {
    NCC_METHOD: ("Frozen NCC", "#205781", "o", "-"),
    PHASE_METHOD: ("Frozen phase", "#D1762A", "s", "-"),
    "ncc_prebias_same_gate_diagnostic": (
        "NCC pre-bias diagnostic", "#4382AF", "o", "--"
    ),
    "phase_prebias_same_gate_diagnostic": (
        "Phase pre-bias diagnostic", "#E7A268", "s", "--"
    ),
    IDENTITY: ("No correction", "#666666", "^", ":")
}

fig, ax = plt.subplots(figsize=(8.5, 5.2))

for method, (label, color, marker, linestyle) in styles.items():

    sub = shift_group[
        shift_group["method"] == method
    ].sort_values("shift_magnitude")

    ax.plot(
        sub["shift_magnitude"],
        sub["mean_epe"],
        label=label,
        color=color,
        marker=marker,
        linestyle=linestyle,
        linewidth=1.9,
        markersize=5.5
    )

ax.set_xlabel("Applied displacement magnitude (pixels)")
ax.set_ylabel("Mean endpoint error (pixels)")

ax.set_xticks(
    sorted(shift_group["shift_magnitude"].unique())
)

ax.grid(axis="y", alpha=0.22)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, fontsize=9, ncol=2)

fig.tight_layout()

fig.savefig(
    OUT / "Step42_Frozen_vs_Counterfactual.png",
    dpi=900, bbox_inches="tight", facecolor="white"
)

fig.savefig(
    OUT / "Step42_Frozen_vs_Counterfactual.pdf",
    bbox_inches="tight", facecolor="white"
)

plt.show()
plt.close(fig)

# -------------------------------------------------------------------------
# H — FIGURE 2: ALIGNED RESIDUAL VECTOR DISTRIBUTION
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.5, 6))

vector_styles = [
    (NCC_METHOD, "#205781", "o", "Frozen NCC"),
    (PHASE_METHOD, "#D1762A", "s", "Frozen phase"),
    (
        "ncc_prebias_same_gate_diagnostic",
        "#4382AF", "o", "NCC pre-bias diagnostic"
    ),
    (
        "phase_prebias_same_gate_diagnostic",
        "#E7A268", "s", "Phase pre-bias diagnostic"
    )
]

for method, color, marker, label in vector_styles:

    sub = aligned[
        aligned["method"] == method
    ]

    ax.scatter(
        sub["signed_error_dx"],
        sub["signed_error_dy"],
        s=64, color=color,
        marker=marker,
        edgecolor="white",
        linewidth=0.5,
        alpha=0.83,
        label=label,
        zorder=3
    )

ax.scatter(
    [0], [0],
    marker="+",
    color="black",
    s=120,
    linewidth=1.7,
    label="Zero displacement",
    zorder=5
)

ax.axhline(0, color="#888888", linewidth=0.8)
ax.axvline(0, color="#888888", linewidth=0.8)

ax.set_xlabel("Horizontal residual, dx (pixels)")
ax.set_ylabel("Vertical residual, dy (pixels)")

ax.set_aspect("equal", adjustable="datalim")
ax.grid(alpha=0.13)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False,
    fontsize=8,
    loc="best"
)

fig.tight_layout()

fig.savefig(
    OUT / "Step42_Aligned_Residual_Vectors.png",
    dpi=900, bbox_inches="tight", facecolor="white"
)

fig.savefig(
    OUT / "Step42_Aligned_Residual_Vectors.pdf",
    bbox_inches="tight", facecolor="white"
)

plt.show()
plt.close(fig)

# -------------------------------------------------------------------------
# I — FINAL REPORT
# -------------------------------------------------------------------------

report = {
    "experiment": (
        "External MDAS frozen-bias transfer attribution"
    ),
    "status": "POST_HOC_CALIBRATION_TRANSFER_AUDIT_COMPLETED",
    "source_step41c_results_sha256": sha256(INPUT_RESULTS),
    "source_step41c_report_sha256": sha256(INPUT_REPORT),
    "source_acquisitions": 1,
    "patches": 5,
    "conditions_per_patch": 17,
    "frozen_method_records": 255,
    "diagnostic_counterfactual_records": 170,
    "historical_ncc_bias": [
        NCC_BIAS_DY, NCC_BIAS_DX
    ],
    "historical_phase_bias": [
        PHASE_BIAS_DY, PHASE_BIAS_DX
    ],
    "original_ncc_gate": NCC_GATE,
    "original_phase_gate": PHASE_GATE,
    "summary": summary.to_dict(orient="records"),
    "paired_mean_diagnostic_epe_reductions": {
        family: float(
            group["diagnostic_epe_reduction"].mean()
        )
        for family, group in attribution.groupby(
            "method_family"
        )
    },
    "registration_estimators_rerun": False,
    "frozen_registration_parameters_modified": False,
    "independent_external_improvement_claim_supported": False,
    "external_fusion_models_evaluated": False,
    "interpretation": [
        "The frozen methods are the primary external results.",
        "Bias-neutral estimates are post hoc counterfactuals.",
        "Any apparent improvement cannot be reported as "
        "independent validation of a new method.",
        "A small number of patches from one acquisition "
        "does not establish broad generalization.",
        "The external sensor interface and synthetic MSI "
        "differ from the original MDAS training domain."
    ],
    "output_directory": str(OUT)
}

save_json(
    OUT / "Step42_Final_Report.json",
    report
)

print("\n" + "=" * 88)
print("STEP 42 — FINAL REPORT")
print("=" * 88)
print(json.dumps(report, indent=2))

print("\nSTEP 42 COMPLETE")
print("All Step 41C results and frozen configurations preserved.")


In [ ]:

# =============================================================================
# STEP 43 — MDAS FINAL EXTERNAL VALIDATION RESULTS PACKAGE
#
# Purpose:
#   Consolidate frozen external transfer and exploratory bias attribution
#   without mixing confirmatory and post hoc results.
#
# Sources:
#   Step41C original frozen Potsdam transfer
#   Step42 external calibration-transfer diagnostic
#   Step40 historical reproduction verification
#
# Outputs:
#   Step43_Table_External_Performance.csv
#   Step43_Table_Patch_Level.csv
#   Step43_Table_Bias_Attribution.csv
#   Step43_Figure_External_EPE.png/.pdf
#   Step43_Figure_Aligned_Residuals.png/.pdf
#   Step43_Figure_Captions.txt
#   Step43_Final_Report.json
#
# Figure requirements:
#   - White background, no decorative titles
#   - Print-readable fonts and aligned axes
#   - PNG at 900 dpi and vector PDF
#   - No confidence intervals treating 85 shifts as independent scenes
#
# Does not retrain, rerun registration, or modify source outputs.
# =============================================================================

from google.colab import drive
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

drive.mount("/content/drive")

P = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

S40 = P / "Step40_Reproduction_Discrepancy_Audit"
S41 = P / "Step41C_Frozen_Potsdam_Transfer"
S42 = P / "Step42_External_Calibration_Transfer_Audit"

OUT = P / "Step43_External_Manuscript_Results"
OUT.mkdir(parents=True, exist_ok=True)

outputs = [
    "Step43_Table_External_Performance.csv",
    "Step43_Table_Patch_Level.csv",
    "Step43_Table_Bias_Attribution.csv",
    "Step43_Figure_External_EPE.png",
    "Step43_Figure_External_EPE.pdf",
    "Step43_Figure_Aligned_Residuals.png",
    "Step43_Figure_Aligned_Residuals.pdf",
    "Step43_Figure_Captions.txt",
    "Step43_Final_Report.json"
]

existing = [x for x in outputs if (OUT / x).exists()]
if existing:
    raise FileExistsError(
        "Previous Step43 outputs exist; preserving them:\n"
        + "\n".join(existing)
    )

def digest(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

sources = {
    "historical_reproduction":
        S40 / "Step40_Final_Report.json",
    "frozen_results":
        S41 / "Step41C_Registration_Results.csv",
    "frozen_report":
        S41 / "Step41C_Final_Report.json",
    "diagnostic_results":
        S42 / "Step42_Bias_Attribution_Cases.csv",
    "diagnostic_report":
        S42 / "Step42_Final_Report.json"
}

for name, path in sources.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")

r40 = read_json(sources["historical_reproduction"])
r41 = read_json(sources["frozen_report"])
r42 = read_json(sources["diagnostic_report"])

assert r40["status"] == "NUMERICAL_REPRODUCTION_CONSISTENT"
assert r41["status"] == (
    "EXTERNAL_FROZEN_REGISTRATION_EVALUATION_COMPLETE"
)
assert r42["status"] == (
    "POST_HOC_CALIBRATION_TRANSFER_AUDIT_COMPLETED"
)

# Confirm Step42 is tied to the exact same original Step41C CSV.
assert r42["source_step41c_results_sha256"] == digest(
    sources["frozen_results"]
)

frozen = pd.read_csv(sources["frozen_results"])
combined = pd.read_csv(sources["diagnostic_results"])

assert len(frozen) == 255
assert len(combined) == 425

KEYS = ["patch_id", "condition_index"]

METHODS = {
    "frozen_ncc_step13": {
        "label": "NCC, frozen",
        "status": "Primary frozen transfer",
        "color": "#205781"
    },
    "frozen_phase_step20_size_adapted": {
        "label": "Phase, frozen",
        "status": "Primary frozen transfer",
        "color": "#C16C25"
    },
    "identity": {
        "label": "No correction",
        "status": "Baseline",
        "color": "#707070"
    },
    "ncc_prebias_same_gate_diagnostic": {
        "label": "NCC, bias omitted",
        "status": "Post hoc diagnostic",
        "color": "#6199C3"
    },
    "phase_prebias_same_gate_diagnostic": {
        "label": "Phase, bias omitted",
        "status": "Post hoc diagnostic",
        "color": "#E0A371"
    }
}

if set(combined["method"]) != set(METHODS):
    raise ValueError(
        "Unexpected Step42 methods: "
        + str(sorted(combined["method"].unique()))
    )

if combined.duplicated(KEYS + ["method"]).any():
    raise RuntimeError("Duplicate experimental condition keys")

assert combined["patch_id"].nunique() == 5
assert (combined.groupby("method").size() == 85).all()

for column in [
    "true_dy", "true_dx",
    "estimated_dy", "estimated_dx", "epe"
]:
    combined[column] = pd.to_numeric(
        combined[column], errors="raise"
    )

assert np.isfinite(
    combined[
        ["true_dy", "true_dx",
         "estimated_dy", "estimated_dx", "epe"]
    ].to_numpy(float)
).all()

# Independently verify the EPE calculations.
calculated_epe = np.hypot(
    combined["estimated_dy"] - combined["true_dy"],
    combined["estimated_dx"] - combined["true_dx"]
)

if not np.allclose(
    calculated_epe, combined["epe"], rtol=0, atol=1e-10
):
    raise RuntimeError("Stored endpoint error mismatch")

# Check original frozen results are identical across steps.
for method in [
    "frozen_ncc_step13",
    "frozen_phase_step20_size_adapted",
    "identity"
]:
    a = frozen[frozen["method"] == method].sort_values(KEYS)
    b = combined[combined["method"] == method].sort_values(KEYS)

    if not np.allclose(
        a["epe"].to_numpy(float),
        b["epe"].to_numpy(float),
        atol=0,
        rtol=0
    ):
        raise RuntimeError(
            f"Original frozen EPE changed: {method}"
        )

print("=" * 88)
print("STEP 43 — VERIFIED MANUSCRIPT RESULTS PACKAGE")
print("=" * 88)
print("Historical reproduction: PASSED")
print("Frozen experiment: 255 records")
print("Post hoc diagnostics: 170 records")
print("Independent external acquisitions: 1")
print("Spatially distinct patches: 5")

# -------------------------------------------------------------------------
# 43A — MAIN EXTERNAL PERFORMANCE TABLE
# -------------------------------------------------------------------------

table_rows = []

for method, values in METHODS.items():
    group = combined[combined["method"] == method]
    aligned = group[
        (group["true_dy"] == 0)
        & (group["true_dx"] == 0)
    ]

    table_rows.append({
        "method": values["label"],
        "method_id": method,
        "evidence_class": values["status"],
        "source_scenes": 1,
        "patches": int(group["patch_id"].nunique()),
        "conditions": int(len(group)),
        "mean_epe_px": float(group["epe"].mean()),
        "median_epe_px": float(group["epe"].median()),
        "maximum_epe_px": float(group["epe"].max()),
        "within_025_pct": float(
            100 * (group["epe"] <= 0.25).mean()
        ),
        "within_050_pct": float(
            100 * (group["epe"] <= 0.5).mean()
        ),
        "aligned_preserved": int((
            (aligned["estimated_dy"] == 0)
            & (aligned["estimated_dx"] == 0)
        ).sum()),
        "aligned_cases": int(len(aligned)),
        "aligned_mean_epe_px": float(
            aligned["epe"].mean()
        )
    })

table = pd.DataFrame(table_rows)
table.to_csv(
    OUT / "Step43_Table_External_Performance.csv",
    index=False
)

print("\n43A — EXTERNAL REGISTRATION TABLE")
print(
    table[
        [
            "method", "evidence_class", "mean_epe_px",
            "within_025_pct", "aligned_preserved"
        ]
    ].round(4).to_string(index=False)
)

# -------------------------------------------------------------------------
# 43B — PATCH-LEVEL RESULTS
# -------------------------------------------------------------------------

patch_table = (
    combined
    .groupby(["patch_id", "method"], as_index=False)
    .agg(
        conditions=("epe", "size"),
        mean_epe_px=("epe", "mean"),
        maximum_epe_px=("epe", "max")
    )
)

patch_table["evidence_class"] = (
    patch_table["method"]
    .map(lambda m: METHODS[m]["status"])
)

patch_table.to_csv(
    OUT / "Step43_Table_Patch_Level.csv",
    index=False
)

# -------------------------------------------------------------------------
# 43C — PAIRED FROZEN VERSUS BIAS-OMITTED DIFFERENCES
# -------------------------------------------------------------------------

pairs = [
    (
        "frozen_ncc_step13",
        "ncc_prebias_same_gate_diagnostic",
        "NCC"
    ),
    (
        "frozen_phase_step20_size_adapted",
        "phase_prebias_same_gate_diagnostic",
        "Phase correlation"
    )
]

attribution_rows = []

for frozen_id, diagnostic_id, label in pairs:
    a = combined[
        combined["method"] == frozen_id
    ][KEYS + ["epe"]].rename(
        columns={"epe": "frozen_epe"}
    )

    b = combined[
        combined["method"] == diagnostic_id
    ][KEYS + ["epe"]].rename(
        columns={"epe": "diagnostic_epe"}
    )

    comparison = a.merge(
        b, on=KEYS, validate="one_to_one"
    )

    comparison["difference"] = (
        comparison["frozen_epe"]
        - comparison["diagnostic_epe"]
    )

    patch_differences = (
        comparison
        .groupby("patch_id")["difference"]
        .mean()
    )

    attribution_rows.append({
        "family": label,
        "paired_conditions": len(comparison),
        "independent_source_scenes": 1,
        "patches": len(patch_differences),
        "frozen_mean_epe_px": float(
            comparison["frozen_epe"].mean()
        ),
        "diagnostic_mean_epe_px": float(
            comparison["diagnostic_epe"].mean()
        ),
        "posthoc_mean_reduction_px": float(
            comparison["difference"].mean()
        ),
        "patches_with_positive_reduction": int(
            (patch_differences > 0).sum()
        ),
        "interpretation": "Exploratory, not independent validation"
    })

attribution_table = pd.DataFrame(attribution_rows)

attribution_table.to_csv(
    OUT / "Step43_Table_Bias_Attribution.csv",
    index=False
)

print("\n43B — BIAS ATTRIBUTION")
print(
    attribution_table[
        [
            "family",
            "frozen_mean_epe_px",
            "diagnostic_mean_epe_px",
            "posthoc_mean_reduction_px",
            "patches_with_positive_reduction"
        ]
    ].round(5).to_string(index=False)
)

# -------------------------------------------------------------------------
# 43D — FIGURE 1: MEAN EPE BY METHOD
# -------------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 10,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.facecolor": "white"
})

fig, ax = plt.subplots(figsize=(9.3, 4.6))

order = [
    "frozen_ncc_step13",
    "frozen_phase_step20_size_adapted",
    "identity",
    "ncc_prebias_same_gate_diagnostic",
    "phase_prebias_same_gate_diagnostic"
]

plot_table = table.set_index("method_id").loc[order]

x = np.arange(len(plot_table))

bars = ax.bar(
    x,
    plot_table["mean_epe_px"],
    width=0.62,
    color=[METHODS[m]["color"] for m in order],
    edgecolor="#353535",
    linewidth=0.5
)

for i, bar in enumerate(bars):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.035,
        f"{plot_table['mean_epe_px'].iloc[i]:.3f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

ax.set_xticks(x)
ax.set_xticklabels(
    [METHODS[m]["label"] for m in order],
    rotation=18,
    ha="right"
)

ax.set_ylabel("Mean endpoint error (pixels)")
ax.set_ylim(
    0,
    float(plot_table["mean_epe_px"].max()) * 1.22
)

ax.axvline(
    2.5, color="#777777",
    linewidth=0.9,
    linestyle="--"
)

ax.text(
    1.0, 0.97,
    "Frozen transfer and baseline",
    transform=ax.get_xaxis_transform(),
    ha="center", va="top", fontsize=9
)

ax.text(
    3.5, 0.97,
    "Post hoc diagnostics",
    transform=ax.get_xaxis_transform(),
    ha="center", va="top", fontsize=9
)

ax.grid(
    axis="y", alpha=0.18,
    linewidth=0.7
)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout(pad=1.1)

fig.savefig(
    OUT / "Step43_Figure_External_EPE.png",
    dpi=900,
    bbox_inches="tight",
    facecolor="white"
)
fig.savefig(
    OUT / "Step43_Figure_External_EPE.pdf",
    bbox_inches="tight",
    facecolor="white"
)
plt.show()
plt.close(fig)

# -------------------------------------------------------------------------
# 43E — FIGURE 2: ALIGNED RESIDUALS
# -------------------------------------------------------------------------

aligned = combined[
    (combined["true_dy"] == 0)
    & (combined["true_dx"] == 0)
].copy()

fig, ax = plt.subplots(figsize=(6.7, 6.0))

for method in [
    "frozen_ncc_step13",
    "frozen_phase_step20_size_adapted",
    "ncc_prebias_same_gate_diagnostic",
    "phase_prebias_same_gate_diagnostic"
]:
    group = aligned[
        aligned["method"] == method
    ]

    ax.scatter(
        group["estimated_dx"],
        group["estimated_dy"],
        s=60,
        color=METHODS[method]["color"],
        edgecolor="white",
        linewidth=0.6,
        alpha=0.85,
        label=METHODS[method]["label"]
    )

ax.scatter(
    [0], [0], marker="+",
    color="black", s=130,
    linewidth=1.7, zorder=6,
    label="Correct alignment"
)

ax.axhline(0, color="#777777", lw=0.8)
ax.axvline(0, color="#777777", lw=0.8)

ax.set_xlabel("Horizontal residual, dx (pixels)")
ax.set_ylabel("Vertical residual, dy (pixels)")
ax.set_aspect("equal", adjustable="box")

# Fixed descriptive limits derived from all aligned residuals.
low = float(min(
    aligned["estimated_dx"].min(),
    aligned["estimated_dy"].min(),
    0
))
high = float(max(
    aligned["estimated_dx"].max(),
    aligned["estimated_dy"].max(),
    0
))

pad = max(0.12, (high - low) * 0.15)
ax.set_xlim(low - pad, high + pad)
ax.set_ylim(low - pad, high + pad)

ax.grid(alpha=0.16)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False,
    fontsize=8,
    loc="upper left"
)

fig.tight_layout()

fig.savefig(
    OUT / "Step43_Figure_Aligned_Residuals.png",
    dpi=900,
    bbox_inches="tight",
    facecolor="white"
)
fig.savefig(
    OUT / "Step43_Figure_Aligned_Residuals.pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

# -------------------------------------------------------------------------
# 43F — FIGURE CAPTIONS
# -------------------------------------------------------------------------

captions = """Figure 1. Registration endpoint error on the Potsdam
synthetic HSI–MSI benchmark for the frozen MDAS methods,
the no-correction baseline, and post hoc bias-omission
diagnostics. Each method was evaluated under 85 synthetic
translation conditions across five non-overlapping patches
from one EnMAP acquisition. The frozen methods represent
the primary transfer experiment. The bias-omission results
are exploratory counterfactuals examined after the external
transfer outcomes were known.

Figure 2. Estimated residual displacement vectors for the
five truly aligned Potsdam patch observations. Residuals
are shown for both frozen MDAS registration methods and
their post hoc bias-omission diagnostics. The origin denotes
correctly preserved alignment. All five patches originate
from a single EnMAP acquisition; the plotted observations
must not be interpreted as independent geographic scenes.
"""

(OUT / "Step43_Figure_Captions.txt").write_text(
    captions,
    encoding="utf-8"
)

# -------------------------------------------------------------------------
# 43G — FINAL REPORT
# -------------------------------------------------------------------------

report = {
    "status": "EXTERNAL_MANUSCRIPT_RESULTS_PACKAGE_COMPLETE",
    "historical_registration_reproduced": True,
    "external_acquisitions": 1,
    "external_patches": 5,
    "synthetic_conditions_per_patch": 17,
    "frozen_method_case_records": 255,
    "posthoc_diagnostic_case_records": 170,
    "primary_analysis_methods": [
        "frozen_ncc_step13",
        "frozen_phase_step20_size_adapted",
        "identity"
    ],
    "exploratory_methods": [
        "ncc_prebias_same_gate_diagnostic",
        "phase_prebias_same_gate_diagnostic"
    ],
    "source_sha256": {
        name: digest(path)
        for name, path in sources.items()
    },
    "external_result_table": table.to_dict("records"),
    "bias_attribution": attribution_table.to_dict("records"),
    "inferential_statistics_performed": False,
    "source_scene_replicates": 1,
    "fusion_checkpoints_evaluated_on_potsdam": False,
    "scientific_conclusion": (
        "The original registration estimators reproduce historical "
        "MDAS records. Transferring their frozen calibration offsets "
        "to the Potsdam synthetic benchmark introduces systematic "
        "alignment errors. Bias-omission counterfactuals suggest "
        "calibration portability is a major cause; their favorable "
        "performance is exploratory rather than independently validated."
    ),
    "limitations": [
        "One independent EnMAP acquisition",
        "Dependent synthetic perturbations of five spatial patches",
        "Synthetic rather than real multispectral sensor observations",
        "Necessary spectral-interface and phase-size adaptations",
        "Post hoc bias-neutral diagnostics",
        "No external fusion-model quality measurements"
    ],
    "output_directory": str(OUT)
}

(OUT / "Step43_Final_Report.json").write_text(
    json.dumps(report, indent=2, allow_nan=False),
    encoding="utf-8"
)

print("\n" + "=" * 88)
print("STEP 43 — FINAL REPORT")
print("=" * 88)

print(json.dumps({
    "status": report["status"],
    "source_scene_replicates": 1,
    "primary_methods": 3,
    "posthoc_diagnostic_methods": 2,
    "tables": 3,
    "figures": 2,
    "output_directory": str(OUT)
}, indent=2))

print("\nSTEP 43 COMPLETE")
print("All historical and external results unchanged.")


In [ ]:

# =============================================================================
# STEP 44C — FINAL VERIFIED POTSDAM SPATIAL FIGURES
# =============================================================================
#
# COMPLETE CORRECTED VERSION
#
# Fixes:
#   1. Rasterio band indexes converted to Python list
#   2. Step41C vs Step42 records joined on exact case keys
#   3. CSV numerical differences checked using 1e-9 tolerance
#   4. Spectral band order and image provenance verified
#   5. Patch footprints derived from original GeoTIFF transform
#   6. Common image stretches and error color scales
#   7. Frozen vs post hoc results identified separately
#
# Figures:
#   F01  Actual Potsdam EnMAP scene and five patch footprints
#   F02  Real MSI registration image comparison (two patches)
#   F03  Four-band pixelwise image difference maps
#   F04  Spatial patch-level mean registration EPE maps
#
# Outputs: 900-dpi PNG + vector PDF, CSV audits, captions, JSON report.
#
# Does NOT:
#   - Execute original registration algorithms again
#   - Retrain fusion checkpoints
#   - Refit SRF weights or bias offsets
#   - Use external ground truth to select new methods
#   - Overwrite any historical output
#
# LIMITATION:
#   All five Potsdam patches are from ONE EnMAP acquisition.
#   Bias-omitted registration results remain POST HOC DIAGNOSTICS.
# =============================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.ndimage import shift as ndi_shift
import rasterio

# =============================================================================
# A — PROJECT PATHS AND OUTPUT PROTECTION
# =============================================================================

START = time.monotonic()

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

S27 = PROJECT / "Step27_External_EnMAP_Pilot"
S30 = PROJECT / "Step30_Frozen_Potsdam_Benchmark"
S41 = PROJECT / "Step41C_Frozen_Potsdam_Transfer"
S42 = PROJECT / "Step42_External_Calibration_Transfer_Audit"

SOURCE_TIF = S27 / "Imagery" / "enmap_potsdam.tif"
PROTOCOL_PATH = S30 / "Step30_Frozen_Forward_Model.json"
RESPONSE_PATH = S30 / "Step30_Spectral_Response_Matrix.csv"

FROZEN_CSV = S41 / "Step41C_Registration_Results.csv"
FROZEN_REPORT = S41 / "Step41C_Final_Report.json"

DIAGNOSTIC_CSV = S42 / "Step42_Bias_Attribution_Cases.csv"
DIAGNOSTIC_REPORT = S42 / "Step42_Final_Report.json"

OUT = PROJECT / "Step44C_Final_Verified_Spatial_Figures"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_OUTPUTS = [
    "F01_EnMAP_Scene_Patches.png",
    "F01_EnMAP_Scene_Patches.pdf",
    "F02_Registration_Image_Examples.png",
    "F02_Registration_Image_Examples.pdf",
    "F03_Four_Band_Difference_Maps.png",
    "F03_Four_Band_Difference_Maps.pdf",
    "F04_Spatial_Patch_Registration_EPE.png",
    "F04_Spatial_Patch_Registration_EPE.pdf",
    "Step44C_Frozen_Comparison_Audit.csv",
    "Step44C_Patch_Footprints.csv",
    "Step44C_Example_Values.csv",
    "Step44C_Patch_EPE.csv",
    "Step44C_Figure_Captions.txt",
    "Step44C_Final_Report.json"
]

already_exists = [
    f for f in EXPECTED_OUTPUTS if (OUT / f).exists()
]

if already_exists:
    raise FileExistsError(
        "Step44C output files already exist. "
        "Do not overwrite earlier results. Change OUT to "
        "a new directory name before rerunning:\n"
        + "\n".join(already_exists)
    )

print("=" * 90)
print("STEP 44C — FINAL VERIFIED POTSDAM SPATIAL FIGURES")
print("=" * 90)


# =============================================================================
# B — HELPERS
# =============================================================================

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()


def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path, obj):
    path.write_text(
        json.dumps(
            obj, indent=2, allow_nan=False, default=str
        ),
        encoding="utf-8"
    )


def require_finite(name, values):
    if not np.isfinite(np.asarray(values)).all():
        raise ValueError(f"Nonfinite values: {name}")


def export_figure(fig, stem):
    png = OUT / (stem + ".png")
    pdf = OUT / (stem + ".pdf")

    fig.savefig(
        png,
        dpi=900,
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.10
    )
    fig.savefig(
        pdf,
        bbox_inches="tight",
        facecolor="white",
        pad_inches=0.10
    )

    print("Saved:", png.name, "and", pdf.name, flush=True)
    plt.show()
    plt.close(fig)


plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
})


# =============================================================================
# C — LOAD AND VERIFY EXPERIMENTAL RECORDS
# =============================================================================

paths = {
    "source_tif": SOURCE_TIF,
    "protocol": PROTOCOL_PATH,
    "response": RESPONSE_PATH,
    "frozen_csv": FROZEN_CSV,
    "frozen_report": FROZEN_REPORT,
    "diagnostic_csv": DIAGNOSTIC_CSV,
    "diagnostic_report": DIAGNOSTIC_REPORT
}

for label, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"{label}: {path}")

protocol = load_json(PROTOCOL_PATH)
r41 = load_json(FROZEN_REPORT)
r42 = load_json(DIAGNOSTIC_REPORT)

frozen = pd.read_csv(FROZEN_CSV)
combined = pd.read_csv(DIAGNOSTIC_CSV)
response_df = pd.read_csv(RESPONSE_PATH)

assert r41["status"] == (
    "EXTERNAL_FROZEN_REGISTRATION_EVALUATION_COMPLETE"
)
assert r42["status"] == (
    "POST_HOC_CALIBRATION_TRANSFER_AUDIT_COMPLETED"
)

if r42["source_step41c_results_sha256"] != sha256(FROZEN_CSV):
    raise RuntimeError(
        "Step42 was derived from a different Step41C CSV."
    )

assert len(frozen) == 255
assert len(combined) == 425
assert protocol["source_scene_count"] == 1
assert protocol["retained_band_count"] == 218

NCC = "frozen_ncc_step13"
PHASE = "frozen_phase_step20_size_adapted"
IDENTITY = "identity"
NCC_DIAG = "ncc_prebias_same_gate_diagnostic"
PHASE_DIAG = "phase_prebias_same_gate_diagnostic"

METHODS = [NCC, PHASE, IDENTITY, NCC_DIAG, PHASE_DIAG]

if set(combined["method"]) != set(METHODS):
    raise RuntimeError(
        "Unexpected methods in diagnostic results."
    )

KEYS = ["patch_id", "condition_index"]

for name, df in [
    ("Step41C", frozen),
    ("Step42", combined)
]:
    if df.duplicated(KEYS + ["method"]).any():
        raise RuntimeError(f"Duplicate records in {name}")

if not (combined.groupby("method").size() == 85).all():
    raise RuntimeError(
        "Expected 85 cases per method."
    )

numeric_fields = [
    "true_dy", "true_dx",
    "estimated_dy", "estimated_dx", "epe"
]

for df in [frozen, combined]:
    for col in numeric_fields:
        df[col] = pd.to_numeric(df[col], errors="raise")
    require_finite("registration records", df[numeric_fields].values)

# Independently recalculate the endpoint errors.
recalculated = np.hypot(
    combined["estimated_dy"] - combined["true_dy"],
    combined["estimated_dx"] - combined["true_dx"]
)

if not np.allclose(
    recalculated,
    combined["epe"],
    atol=1e-10,
    rtol=0
):
    raise RuntimeError(
        "Stored Step42 endpoint errors are inconsistent."
    )

print("\nC — RECORDS VERIFIED")
print("Frozen records:", len(frozen))
print("Post hoc diagnostic records:", len(combined) - len(frozen))
print("Independent source acquisitions:", 1)


# =============================================================================
# D — CORRECTED CROSS-STEP FLOATING-POINT AUDIT
# =============================================================================

# This is the correction for the earlier Step44B failure.
# Compare matching experiment cases, not separately sorted arrays.
# Allow only negligible CSV serialization differences.

COMPARISON_ATOL = 1e-9

audit_rows = []

for method in [NCC, PHASE, IDENTITY]:

    a = frozen[
        frozen["method"] == method
    ][KEYS + numeric_fields].copy()

    b = combined[
        combined["method"] == method
    ][KEYS + numeric_fields].copy()

    if len(a) != 85 or len(b) != 85:
        raise RuntimeError(
            f"Incorrect frozen case count: {method}"
        )

    comparison = a.merge(
        b,
        on=KEYS,
        how="outer",
        suffixes=("_41C", "_42"),
        indicator=True,
        validate="one_to_one"
    )

    if len(comparison) != 85:
        raise RuntimeError(f"Case join failed: {method}")

    if not comparison["_merge"].eq("both").all():
        raise RuntimeError(
            f"Step41C and Step42 have different case keys: {method}"
        )

    for field in numeric_fields:

        original = comparison[
            field + "_41C"
        ].to_numpy(dtype=float)

        saved_copy = comparison[
            field + "_42"
        ].to_numpy(dtype=float)

        difference = np.abs(original - saved_copy)

        require_finite(
            f"{method}/{field}",
            difference
        )

        max_difference = float(difference.max())

        audit_rows.append({
            "method": method,
            "field": field,
            "matched_cases": len(comparison),
            "max_absolute_difference": max_difference,
            "mean_absolute_difference": float(difference.mean()),
            "accepted_tolerance": COMPARISON_ATOL,
            "passed": bool(max_difference <= COMPARISON_ATOL)
        })

        print(
            f"{method} | {field} | "
            f"max difference = {max_difference:.3e}"
        )

audit = pd.DataFrame(audit_rows)

audit.to_csv(
    OUT / "Step44C_Frozen_Comparison_Audit.csv",
    index=False
)

if not audit["passed"].all():
    failures = audit[~audit["passed"]]

    print("\nCROSS-STEP INCONSISTENCIES")
    print(failures.to_string(index=False))

    raise RuntimeError(
        "Step41C and Step42 differ beyond the "
        "permitted 1e-9 CSV tolerance. "
        "Review the saved audit instead of ignoring the difference."
    )

print("\nD — FROZEN RECORD CONSISTENCY: PASSED")
print("All 255 original method-case records reconciled.")


# =============================================================================
# E — LOAD THE ORIGINAL GEOGRAPHIC ENMAP RASTER
# =============================================================================

EXPECTED_TIF_SHA = (
    "4ba14fd27ac80c9169f3d733010dbdaa986584b3a062dc497578c3e114733329"
)

if sha256(SOURCE_TIF) != EXPECTED_TIF_SHA:
    raise RuntimeError(
        "EnMAP source TIFF does not match the recorded source hash."
    )

required_response_columns = [
    "original_hsi_band",
    "wavelength_nm",
    "synthetic_blue_weight",
    "synthetic_green_weight",
    "synthetic_red_weight",
    "synthetic_nir_weight"
]

if not set(required_response_columns).issubset(
    response_df.columns
):
    raise RuntimeError(
        "Unexpected Step30 spectral response schema."
    )

retained_ids = np.asarray(
    protocol["retained_band_ids_1_based"],
    dtype=int
)

source_band_ids = response_df[
    "original_hsi_band"
].to_numpy(dtype=int)

if not np.array_equal(retained_ids, source_band_ids):
    raise RuntimeError("Spectral band ordering mismatch.")

# Correct Rasterio indexing: Python list, not NumPy array.
band_indexes = [int(b) for b in retained_ids.tolist()]

with rasterio.open(SOURCE_TIF) as src:

    if (src.count, src.height, src.width) != (224, 200, 330):
        raise RuntimeError(
            "Unexpected original EnMAP image dimensions."
        )

    if src.crs is None or not src.crs.is_projected:
        raise RuntimeError(
            "Expected a projected georeferenced EnMAP raster."
        )

    TRANSFORM = src.transform
    BOUNDS = src.bounds
    CRS = str(src.crs)

    if not (
        np.isclose(TRANSFORM.b, 0)
        and np.isclose(TRANSFORM.d, 0)
        and TRANSFORM.a > 0
        and TRANSFORM.e < 0
    ):
        raise RuntimeError(
            "This plotting routine requires a north-up raster."
        )

    cube = src.read(
        indexes=band_indexes
    ).astype(np.float32)

assert cube.shape == (218, 200, 330)
require_finite("EnMAP spectral cube", cube)

weights = response_df[
    [
        "synthetic_blue_weight",
        "synthetic_green_weight",
        "synthetic_red_weight",
        "synthetic_nir_weight"
    ]
].to_numpy(np.float64).T

if weights.shape != (4, 218):
    raise RuntimeError("Unexpected response matrix dimensions.")

if not np.allclose(
    weights.sum(axis=1),
    1,
    atol=1e-7,
    rtol=0
):
    raise RuntimeError("Frozen spectral weights are not normalized.")

# Original saved response weights; nothing is refitted.
scene_msi = np.einsum(
    "bc,chw->bhw",
    weights,
    cube.astype(np.float64),
    optimize=True
).astype(np.float32)

# Synthetic channel order: blue, green, red, NIR.
# Display RGB order: red, green, blue.
scene_rgb_raw = np.moveaxis(
    scene_msi[[2, 1, 0]],
    0,
    -1
)

def get_limits(rgb):
    return (
        np.percentile(rgb, 2, axis=(0, 1)),
        np.percentile(rgb, 98, axis=(0, 1))
    )

def rgb_stretch(rgb, limits):
    lo, hi = limits
    out = (rgb - lo) / np.maximum(hi - lo, 1e-12)
    return np.clip(out, 0, 1) ** 0.85

scene_display = rgb_stretch(
    scene_rgb_raw,
    get_limits(scene_rgb_raw)
)

XMIN, XMAX = BOUNDS.left / 1000, BOUNDS.right / 1000
YMIN, YMAX = BOUNDS.bottom / 1000, BOUNDS.top / 1000

print("\nE — ACTUAL ENMAP RASTER VERIFIED")
print("Original size:", (224, 200, 330))
print("Retained bands:", cube.shape[0])
print("CRS:", CRS)
print("Display RGB size:", scene_display.shape)


# =============================================================================
# F — BUILD TRUE GEOGRAPHIC PATCH FOOTPRINTS
# =============================================================================

PATCH_SIZE = 96

def parse_patch_id(pid):
    match = re.fullmatch(
        r"potsdam_r(\d+)_c(\d+)",
        str(pid)
    )
    if match is None:
        raise ValueError(f"Invalid patch ID: {pid}")
    return int(match.group(1)), int(match.group(2))

PATCH_IDS = sorted(
    protocol["selected_patch_ids"],
    key=parse_patch_id
)

if len(PATCH_IDS) != 5:
    raise RuntimeError("Expected five Potsdam patches.")

def patch_extent(pid):
    row, col = parse_patch_id(pid)
    x0, y0 = TRANSFORM * (col, row)
    x1, y1 = TRANSFORM * (
        col + PATCH_SIZE,
        row + PATCH_SIZE
    )
    return (
        min(x0, x1) / 1000,
        max(x0, x1) / 1000,
        min(y0, y1) / 1000,
        max(y0, y1) / 1000
    )

footprint_rows = []

for i, pid in enumerate(PATCH_IDS, start=1):

    row, col = parse_patch_id(pid)

    if (
        row < 0 or col < 0
        or row + PATCH_SIZE > 200
        or col + PATCH_SIZE > 330
    ):
        raise RuntimeError(
            f"Patch outside original scene: {pid}"
        )

    x0, x1, y0, y1 = patch_extent(pid)

    footprint_rows.append({
        "label": f"P{i}",
        "patch_id": pid,
        "row_start": row,
        "col_start": col,
        "row_end_exclusive": row + PATCH_SIZE,
        "col_end_exclusive": col + PATCH_SIZE,
        "xmin_km": x0,
        "xmax_km": x1,
        "ymin_km": y0,
        "ymax_km": y1,
        "crs": CRS
    })

for i, pid1 in enumerate(PATCH_IDS):
    r1, c1 = parse_patch_id(pid1)
    for pid2 in PATCH_IDS[i+1:]:
        r2, c2 = parse_patch_id(pid2)
        overlaps = (
            r1 < r2 + PATCH_SIZE
            and r2 < r1 + PATCH_SIZE
            and c1 < c2 + PATCH_SIZE
            and c2 < c1 + PATCH_SIZE
        )
        if overlaps:
            raise RuntimeError(
                f"Overlapping patches: {pid1}, {pid2}"
            )

pd.DataFrame(footprint_rows).to_csv(
    OUT / "Step44C_Patch_Footprints.csv",
    index=False
)

def scene_axes(ax):
    ax.imshow(
        scene_display,
        extent=[XMIN, XMAX, YMIN, YMAX],
        origin="upper",
        interpolation="nearest"
    )
    ax.set_xlim(XMIN, XMAX)
    ax.set_ylim(YMIN, YMAX)
    ax.set_aspect("equal")
    ax.set_xlabel("Easting (km)")
    ax.set_ylabel("Northing (km)")
    ax.tick_params(top=True, right=True, direction="out")

def draw_patches(ax, patch_values=None, norm=None, cmap=None):
    for i, pid in enumerate(PATCH_IDS, 1):
        x0, x1, y0, y1 = patch_extent(pid)

        face = (
            "none"
            if patch_values is None
            else cmap(norm(patch_values[pid]))
        )
        edge = "#F8EEA1" if patch_values is None else "white"

        ax.add_patch(Rectangle(
            (x0, y0),
            x1-x0,
            y1-y0,
            facecolor=face,
            edgecolor=edge,
            linewidth=1.4,
            alpha=1.0 if patch_values is None else 0.87,
            zorder=4
        ))

        label = f"P{i}"
        if patch_values is not None:
            label += f"\n{patch_values[pid]:.2f}"

        ax.text(
            (x0+x1)/2,
            (y0+y1)/2,
            label,
            fontsize=8,
            fontweight="bold",
            ha="center",
            va="center",
            color="black",
            bbox=dict(
                facecolor="white",
                edgecolor="none",
                alpha=0.87,
                pad=1.5
            ),
            zorder=6
        )

def add_north_arrow(ax):
    ax.annotate(
        "N",
        xy=(0.95, 0.96),
        xytext=(0.95, 0.83),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        color="white",
        fontweight="bold",
        fontsize=11,
        arrowprops=dict(
            arrowstyle="-|>",
            color="white",
            lw=1.8
        ),
        zorder=10
    )

def add_scale_bar(ax, length_km=1.0):
    width = XMAX-XMIN
    height = YMAX-YMIN

    x = XMIN + 0.07*width
    y = YMIN + 0.07*height
    half = length_km/2
    bar_height = 0.012*height

    for i, color in enumerate(["black", "white"]):
        ax.add_patch(Rectangle(
            (x+i*half, y),
            half,
            bar_height,
            facecolor=color,
            edgecolor="black",
            linewidth=0.8,
            zorder=8
        ))

    ax.text(
        x, y+bar_height+0.015*height,
        "0",
        ha="center",
        fontsize=8,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.85)
    )
    ax.text(
        x+length_km,
        y+bar_height+0.015*height,
        f"{length_km:g} km",
        ha="center",
        fontsize=8,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.85)
    )

# =============================================================================
# F01 — ACTUAL ENMAP SCENE + PATCH LOCATIONS
# =============================================================================

print("\nF01 — ENMAP SCENE AND PATCH FOOTPRINTS", flush=True)

fig, ax = plt.subplots(figsize=(10.2, 6.0))
scene_axes(ax)
draw_patches(ax)
add_north_arrow(ax)
add_scale_bar(ax)

ax.text(
    0.02, 0.97,
    "(a)",
    transform=ax.transAxes,
    va="top",
    fontsize=13,
    fontweight="bold",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.85
    )
)

fig.tight_layout()
export_figure(fig, "F01_EnMAP_Scene_Patches")


# =============================================================================
# G — READ SAVED MSI OBSERVATIONS AND REGISTRATION VALUES
# =============================================================================

def load_patch(pid):
    path = S30 / "Paired_Observations" / f"{pid}.npz"

    if not path.is_file():
        raise FileNotFoundError(path)

    with np.load(path, allow_pickle=False) as z:
        ref = z["hrmsi"].copy()
        shifted = z["shifted_msi"].copy()
        offsets = z["shift_offsets_dy_dx"].copy()
        ids = z["retained_band_ids"].copy()

    if ref.shape != (4, 96, 96):
        raise RuntimeError(f"Invalid reference shape for {pid}")
    if shifted.shape != (17, 4, 96, 96):
        raise RuntimeError(f"Invalid shifted data for {pid}")
    if offsets.shape != (17, 2):
        raise RuntimeError(f"Invalid shifts for {pid}")

    if not np.array_equal(ids.astype(int), retained_ids):
        raise RuntimeError(f"Band ID mismatch for {pid}")

    require_finite(pid, ref)
    require_finite(pid, shifted)

    aligned = np.flatnonzero(
        np.all(np.isclose(offsets, 0, atol=1e-10), axis=1)
    )
    if len(aligned) != 1:
        raise RuntimeError(f"Invalid aligned condition: {pid}")

    if not np.array_equal(
        shifted[int(aligned[0])],
        ref
    ):
        raise RuntimeError(f"Aligned source mismatch for {pid}")

    return ref, shifted, offsets

def find_condition(offsets, dy, dx):
    found = np.flatnonzero(
        np.all(
            np.isclose(offsets, [dy, dx], atol=1e-8),
            axis=1
        )
    )
    if len(found) != 1:
        raise RuntimeError(f"Shift ({dy},{dx}) not uniquely found")
    return int(found[0])

def saved_estimate(pid, condition, method):
    matches = combined[
        (combined["patch_id"] == pid)
        & (combined["condition_index"] == condition)
        & (combined["method"] == method)
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Missing estimate for {pid}, {condition}, {method}"
        )
    row = matches.iloc[0]
    return (
        float(row["estimated_dy"]),
        float(row["estimated_dx"]),
        float(row["epe"])
    )

def resample_for_display(observed, dy, dx):
    # Important: negative displacement corrects the observed shift.
    # This is visualization resampling, NOT new registration inference.
    return ndi_shift(
        observed.astype(np.float32),
        shift=(0, -dy, -dx),
        order=1,
        mode="nearest",
        prefilter=False
    )

def msi_rgb(chw, limits):
    rgb = np.moveaxis(chw[[2, 1, 0]], 0, -1)
    return rgb_stretch(rgb, limits)

def limits_for_msi(chw):
    rgb = np.moveaxis(chw[[2, 1, 0]], 0, -1)
    return get_limits(rgb)

# Fixed example selection: first and last spatial patches.
EXAMPLE_PATCHES = [PATCH_IDS[0], PATCH_IDS[-1]]
EXAMPLE_SHIFT = (1.5, 0.0)

CORRECTION_METHODS = [
    (NCC, "Frozen NCC"),
    (NCC_DIAG, "NCC, bias omitted*"),
    (PHASE, "Frozen phase"),
    (PHASE_DIAG, "Phase, bias omitted*")
]

example_audit = []

# =============================================================================
# F02 — REAL REGISTRATION IMAGE EXAMPLES
# =============================================================================

print("\nF02 — ACTUAL MSI REGISTRATION EXAMPLES", flush=True)

fig, axes = plt.subplots(
    2, 6,
    figsize=(15.3, 5.8),
    constrained_layout=True
)

column_titles = [
    "Aligned reference",
    "Displaced MSI",
    "Frozen NCC",
    "NCC, bias omitted*",
    "Frozen phase",
    "Phase, bias omitted*"
]

for r, pid in enumerate(EXAMPLE_PATCHES):

    reference, shifted, offsets = load_patch(pid)
    idx = find_condition(offsets, *EXAMPLE_SHIFT)
    observed = shifted[idx]
    limits = limits_for_msi(reference)

    images = [reference, observed]
    epe_labels = [None, float(np.hypot(*EXAMPLE_SHIFT))]

    for method, title in CORRECTION_METHODS:
        dy, dx, epe = saved_estimate(pid, idx, method)
        corrected = resample_for_display(observed, dy, dx)

        images.append(corrected)
        epe_labels.append(epe)

        example_audit.append({
            "figure": "F02",
            "patch_id": pid,
            "condition_index": idx,
            "method": method,
            "true_dy": EXAMPLE_SHIFT[0],
            "true_dx": EXAMPLE_SHIFT[1],
            "estimated_dy": dy,
            "estimated_dx": dx,
            "saved_epe": epe
        })

    for c, image in enumerate(images):
        ax = axes[r, c]
        ax.imshow(
            msi_rgb(image, limits),
            interpolation="nearest"
        )
        ax.set_xticks([])
        ax.set_yticks([])

        if r == 0:
            ax.set_title(column_titles[c], fontsize=8.4, pad=7)

        if epe_labels[c] is not None:
            ax.text(
                0.035, 0.045,
                f"EPE {epe_labels[c]:.2f} px",
                transform=ax.transAxes,
                fontsize=7.4,
                color="white",
                bbox=dict(
                    facecolor="black",
                    edgecolor="none",
                    alpha=0.72,
                    pad=2
                )
            )

    axes[r, 0].set_ylabel(pid, fontsize=8.1)

fig.supxlabel(
    "* Post hoc bias-omission diagnostic; corrections "
    "resampled for visualization only.",
    fontsize=9
)

export_figure(fig, "F02_Registration_Image_Examples")

pd.DataFrame(example_audit).to_csv(
    OUT / "Step44C_Example_Values.csv",
    index=False
)


# =============================================================================
# F03 — ACTUAL FOUR-BAND PIXELWISE DIFFERENCE MAPS
# =============================================================================

print("\nF03 — FOUR-BAND IMAGE DIFFERENCE MAPS", flush=True)

pid = PATCH_IDS[0]
reference, shifted, offsets = load_patch(pid)
idx = find_condition(offsets, *EXAMPLE_SHIFT)
observed = shifted[idx]

image_columns = [("Displaced MSI", observed)]

for method, label in CORRECTION_METHODS:
    dy, dx, epe = saved_estimate(pid, idx, method)
    image_columns.append(
        (label, resample_for_display(observed, dy, dx))
    )

def four_band_difference(ref, predicted):
    return np.mean(
        np.abs(
            ref.astype(np.float64)
            - predicted.astype(np.float64)
        ),
        axis=0
    )

BORDER = 6

difference_maps = [
    four_band_difference(reference, image)[
        BORDER:-BORDER, BORDER:-BORDER
    ]
    for label, image in image_columns
]

for image in difference_maps:
    if image.shape != (84, 84):
        raise RuntimeError("Unexpected difference-map shape")
    require_finite("MSI difference map", image)

max_color_value = float(
    np.percentile(
        np.concatenate([x.ravel() for x in difference_maps]),
        99.5
    )
)
max_color_value = max(max_color_value, 1e-12)

fig, axes = plt.subplots(
    1, 5,
    figsize=(15, 3.6),
    constrained_layout=True
)

for i, (ax, (name, _), data) in enumerate(
    zip(axes, image_columns, difference_maps)
):
    im = ax.imshow(
        data,
        cmap="magma",
        vmin=0,
        vmax=max_color_value,
        interpolation="nearest"
    )
    ax.set_title(
        f"({chr(97+i)}) {name}",
        fontsize=8.2
    )
    ax.set_xticks([])
    ax.set_yticks([])

cb = fig.colorbar(
    im,
    ax=axes,
    location="right",
    shrink=0.80,
    pad=0.012
)
cb.set_label(
    "Mean absolute four-band signal difference",
    fontsize=9
)

export_figure(fig, "F03_Four_Band_Difference_Maps")


# =============================================================================
# F04 — SPATIAL PATCH-LEVEL REGISTRATION ERROR
# =============================================================================

print("\nF04 — SPATIAL PATCH-LEVEL ENDPOINT ERROR", flush=True)

PLOT_METHODS = [
    (NCC, "Frozen NCC"),
    (PHASE, "Frozen phase"),
    (NCC_DIAG, "NCC, bias omitted*"),
    (PHASE_DIAG, "Phase, bias omitted*")
]

patch_epe = (
    combined[
        combined["method"].isin(
            [m for m, label in PLOT_METHODS]
        )
    ]
    .groupby(["method", "patch_id"], as_index=False)
    .agg(
        cases=("epe", "size"),
        mean_epe=("epe", "mean"),
        median_epe=("epe", "median"),
        max_epe=("epe", "max")
    )
)

if not (patch_epe["cases"] == 17).all():
    raise RuntimeError(
        "Each patch-method pair must have 17 conditions."
    )

patch_epe.to_csv(
    OUT / "Step44C_Patch_EPE.csv",
    index=False
)

lookup = {}

for method, title in PLOT_METHODS:
    sub = patch_epe[patch_epe["method"] == method]
    lookup[method] = dict(
        zip(
            sub["patch_id"],
            sub["mean_epe"].astype(float)
        )
    )
    if set(lookup[method]) != set(PATCH_IDS):
        raise RuntimeError(
            f"Missing patch means for {method}"
        )

vmax = max(1.5, float(patch_epe["mean_epe"].max()))
norm = Normalize(vmin=0, vmax=vmax)
cmap = plt.get_cmap("YlOrRd")

fig, axes = plt.subplots(
    2, 2,
    figsize=(11.6, 8.7),
    constrained_layout=True
)

for i, (ax, (method, title)) in enumerate(
    zip(axes.flat, PLOT_METHODS)
):
    scene_axes(ax)
    draw_patches(
        ax,
        patch_values=lookup[method],
        norm=norm,
        cmap=cmap
    )
    ax.set_title(
        f"({chr(97+i)}) {title}",
        loc="left",
        fontsize=10.5
    )
    ax.set_xlabel("Easting (km)", fontsize=8)
    ax.set_ylabel("Northing (km)", fontsize=8)
    ax.tick_params(labelsize=7)

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cb = fig.colorbar(
    sm,
    ax=axes.ravel().tolist(),
    shrink=0.7,
    pad=0.015
)
cb.set_label(
    "Patch mean endpoint error (pixels)",
    fontsize=9
)

export_figure(fig, "F04_Spatial_Patch_Registration_EPE")


# =============================================================================
# H — CAPTIONS AND FINAL PROVENANCE
# =============================================================================

captions = """
F01. Spatial distribution of five non-overlapping
96 × 96-pixel evaluation patches in the original Potsdam
EnMAP scene. The displayed RGB composite uses the frozen
four-band spectral-response weights from Step 30.
Patch extents were computed from the source GeoTIFF
geotransform and recorded pixel coordinates.

F02. Image-registration examples using actual synthetic
four-channel MSI observations from two Potsdam patches.
For each patch, an imposed 1.5-pixel vertical displacement
is shown alongside image corrections corresponding to
the frozen MDAS NCC and phase estimators and their post hoc
bias-omission diagnostics. Endpoint errors are obtained
from the saved numerical evaluation. Image resampling is
performed for visualization only, not as a new validation
experiment.

F03. Pixelwise mean absolute difference between an aligned
synthetic four-band MSI reference and the shifted or
visually corrected observations for one Potsdam patch.
All panels share the same display color scale and exclude
a six-pixel visualization border. The maps describe
synthetic MSI signal differences, not hyperspectral
fusion reconstruction errors.

F04. Spatial distribution of patch-level mean endpoint
error for frozen original NCC, frozen bounded phase
correlation, and their post hoc bias-omission diagnostics.
Each patch mean aggregates 17 synthetic displacement
conditions. The five non-overlapping patches belong to
one EnMAP acquisition; they do not represent five
independent geographic test scenes. Bias-omission results
are exploratory and were examined after viewing the
frozen external-transfer results.
"""

(OUT / "Step44C_Figure_Captions.txt").write_text(
    captions.strip(),
    encoding="utf-8"
)

report = {
    "experiment": "Verified Potsdam spatial figure generation",
    "status": "SPATIAL_FIGURES_COMPLETE",
    "figures_generated": 4,
    "png_dpi": 900,
    "vector_pdf_exports": 4,
    "source_acquisitions": 1,
    "source_patches": 5,
    "conditions_per_patch": 17,
    "source_crs": CRS,
    "source_image_shape": [224, 200, 330],
    "retained_hsi_bands": 218,
    "synthetic_msi_bands": 4,
    "cross_step_tolerance": COMPARISON_ATOL,
    "cross_step_max_difference": float(
        audit["max_absolute_difference"].max()
    ),
    "historical_registration_algorithms_rerun": False,
    "registration_parameters_tuned": False,
    "fusion_model_inference_performed": False,
    "bias_omission_methods_posthoc": True,
    "pixelwise_maps_type": (
        "four-band synthetic MSI intensity differences"
    ),
    "source_hashes": {
        name: sha256(path)
        for name, path in paths.items()
    },
    "limitations": [
        "Only one independent external EnMAP scene.",
        "Five spatially distinct patches are not independent scenes.",
        "Seventeen shifts per patch are repeated conditions.",
        "The MSI observations and translations are synthetic.",
        "F02/F03 image correction uses visualization resampling.",
        "F03 does not show hyperspectral reconstruction error.",
        "Post hoc bias-omission diagnostics are exploratory.",
        "No external fusion-model evaluation was performed."
    ],
    "elapsed_seconds": round(time.monotonic()-START, 2),
    "output_directory": str(OUT)
}

write_json(
    OUT / "Step44C_Final_Report.json",
    report
)

print("\n" + "=" * 90)
print("STEP 44C — FINAL REPORT")
print("=" * 90)
print(json.dumps(report, indent=2))

print("\nSTEP 44C COMPLETE")
print("Four spatial figures generated from actual Potsdam data.")
print("Original frozen results unchanged.")
print("Output:", OUT)


In [ ]:

# ============================================================
# STEP 44C — DISPLAY ALL FOUR FINAL SPATIAL FIGURES
# ============================================================

from google.colab import drive
from IPython.display import display, Image, Markdown
from pathlib import Path

drive.mount("/content/drive")

FIGURE_DIR = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026/"
    "Step44C_Final_Verified_Spatial_Figures"
)

figures = [
    (
        "Figure F01 — EnMAP Scene and Sampling Patches",
        "F01_EnMAP_Scene_Patches.png"
    ),
    (
        "Figure F02 — Registration Image Examples",
        "F02_Registration_Image_Examples.png"
    ),
    (
        "Figure F03 — Four-Band Difference Maps",
        "F03_Four_Band_Difference_Maps.png"
    ),
    (
        "Figure F04 — Spatial Registration Error Maps",
        "F04_Spatial_Patch_Registration_EPE.png"
    )
]

print("=" * 75)
print("STEP 44C — FINAL SPATIAL FIGURE DISPLAY")
print("=" * 75)

for title, filename in figures:

    path = FIGURE_DIR / filename

    if not path.is_file():
        print(f"MISSING: {path}")
        continue

    display(Markdown(f"### {title}"))

    display(
        Image(
            filename=str(path),
            width=1200
        )
    )

    print(f"File: {filename}")
    print("-" * 75)

print("\nFIGURE DISPLAY COMPLETE")


In [ ]:

# =============================================================================
# STEP 45 — FINAL SPATIAL FIGURE REFINEMENT
# =============================================================================
#
# FIGURE F02:
#   Real Potsdam MSI observations and registration corrections
#   - 2 spatial patches
#   - 6 comparison columns
#   - Enlarged central details
#   - Reference/observation edge overlays
#
# FIGURE F04:
#   Compact 2x2 spatial patch mean-EPE comparison
#   - Actual georeferenced EnMAP background
#   - Five nonoverlapping 96x96 patch footprints
#   - Shared color scale
#   - Frozen and exploratory methods clearly separated
#
# SOURCE DATA:
#   Step27 original EnMAP TIFF
#   Step30 frozen observations / spectral response
#   Step41C frozen registration evaluation
#   Step42 exploratory bias-omission diagnostics
#
# No methods rerun; no estimates or parameters modified.
# Figures are descriptive visualizations of existing experiments.
# =============================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.ndimage import shift as ndi_shift
from scipy.ndimage import sobel
import rasterio

START = time.monotonic()

ROOT = Path("/content/drive/MyDrive")
PROJECT = ROOT / "MDAS_Registration_Aware_Revision_2026"

S27 = PROJECT / "Step27_External_EnMAP_Pilot"
S30 = PROJECT / "Step30_Frozen_Potsdam_Benchmark"
S41 = PROJECT / "Step41C_Frozen_Potsdam_Transfer"
S42 = PROJECT / "Step42_External_Calibration_Transfer_Audit"

TIF = S27 / "Imagery" / "enmap_potsdam.tif"
PROTOCOL_PATH = S30 / "Step30_Frozen_Forward_Model.json"
RESPONSE_PATH = S30 / "Step30_Spectral_Response_Matrix.csv"

FROZEN_CSV = S41 / "Step41C_Registration_Results.csv"
DIAGNOSTIC_CSV = S42 / "Step42_Bias_Attribution_Cases.csv"
STEP42_REPORT = S42 / "Step42_Final_Report.json"

OUT = PROJECT / "Step45_Refined_Spatial_Manuscript_Figures"
OUT.mkdir(parents=True, exist_ok=True)

OUTPUT_NAMES = [
    "F02_Refined_Registration_And_Edge_Overlays.png",
    "F02_Refined_Registration_And_Edge_Overlays.pdf",
    "F04_Refined_Compact_Spatial_EPE.png",
    "F04_Refined_Compact_Spatial_EPE.pdf",
    "Step45_Figure_Data_Audit.csv",
    "Step45_Patch_EPE.csv",
    "Step45_Figure_Captions.txt",
    "Step45_Final_Report.json",
]

existing = [x for x in OUTPUT_NAMES if (OUT / x).exists()]

if existing:
    raise FileExistsError(
        "Step45 outputs already exist. Use a new output "
        "directory rather than overwrite previous results:\n"
        + "\n".join(existing)
    )

def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def load_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def save_json(path, value):
    path.write_text(
        json.dumps(value, indent=2, allow_nan=False, default=str),
        encoding="utf-8"
    )

def save_fig(fig, stem):
    fig.savefig(
        OUT / f"{stem}.png",
        dpi=900, bbox_inches="tight",
        pad_inches=0.08, facecolor="white"
    )
    fig.savefig(
        OUT / f"{stem}.pdf",
        bbox_inches="tight",
        pad_inches=0.08, facecolor="white"
    )
    print("Saved:", stem, "(PNG and PDF)", flush=True)
    plt.show()
    plt.close(fig)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.75,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "savefig.facecolor": "white"
})

print("="*88)
print("STEP 45 — REFINED SCIENTIFIC SPATIAL FIGURES")
print("="*88)

# =============================================================================
# A — PROVENANCE AND CASE INTEGRITY
# =============================================================================

for path in [
    TIF, PROTOCOL_PATH, RESPONSE_PATH,
    FROZEN_CSV, DIAGNOSTIC_CSV, STEP42_REPORT
]:
    if not path.is_file():
        raise FileNotFoundError(path)

protocol = load_json(PROTOCOL_PATH)
r42 = load_json(STEP42_REPORT)

frozen = pd.read_csv(FROZEN_CSV)
combined = pd.read_csv(DIAGNOSTIC_CSV)
response_df = pd.read_csv(RESPONSE_PATH)

assert len(frozen) == 255
assert len(combined) == 425
assert protocol["source_scene_count"] == 1

if r42["source_step41c_results_sha256"] != file_sha(FROZEN_CSV):
    raise RuntimeError("Step42 does not match Step41C source CSV")

NCC = "frozen_ncc_step13"
PHASE = "frozen_phase_step20_size_adapted"
IDENTITY = "identity"
NCC_DIAG = "ncc_prebias_same_gate_diagnostic"
PHASE_DIAG = "phase_prebias_same_gate_diagnostic"

METHODS = [NCC, PHASE, IDENTITY, NCC_DIAG, PHASE_DIAG]

assert set(combined["method"]) == set(METHODS)
assert (combined.groupby("method").size() == 85).all()

KEYS = ["patch_id", "condition_index"]

if combined.duplicated(KEYS + ["method"]).any():
    raise RuntimeError("Duplicated registration records")

numeric_cols = [
    "true_dy", "true_dx",
    "estimated_dy", "estimated_dx", "epe"
]

for df in [frozen, combined]:
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="raise")
    if not np.isfinite(df[numeric_cols].to_numpy(float)).all():
        raise RuntimeError("Nonfinite saved results")

# Match frozen results by case keys, avoiding row-order assumptions.
for method in [NCC, PHASE, IDENTITY]:
    a = frozen[frozen["method"] == method][KEYS + numeric_cols]
    b = combined[combined["method"] == method][KEYS + numeric_cols]

    matched = a.merge(
        b, on=KEYS,
        suffixes=("_frozen", "_diagnostic_copy"),
        validate="one_to_one"
    )

    if len(matched) != 85:
        raise RuntimeError("Frozen case keys do not reconcile")

    for col in numeric_cols:
        difference = np.abs(
            matched[f"{col}_frozen"].to_numpy(float)
            - matched[f"{col}_diagnostic_copy"].to_numpy(float)
        )
        if float(difference.max()) > 1e-9:
            raise RuntimeError(
                f"Cross-step result changed: {method} / {col}"
            )

print("\n45A — Frozen records verified unchanged")

# =============================================================================
# B — DISPLAY FUNCTIONS AND FROZEN OBSERVATIONS
# =============================================================================

def patch_row_col(pid):
    match = re.fullmatch(r"potsdam_r(\d+)_c(\d+)", pid)
    if match is None:
        raise ValueError(f"Bad patch identifier: {pid}")
    return int(match.group(1)), int(match.group(2))

PATCH_IDS = sorted(
    protocol["selected_patch_ids"], key=patch_row_col
)
assert len(PATCH_IDS) == 5

retained_ids = np.asarray(
    protocol["retained_band_ids_1_based"], dtype=int
)

assert retained_ids.shape == (218,)
assert np.array_equal(
    response_df["original_hsi_band"].to_numpy(int),
    retained_ids
)

RESPONSE = response_df[
    [
        "synthetic_blue_weight",
        "synthetic_green_weight",
        "synthetic_red_weight",
        "synthetic_nir_weight"
    ]
].to_numpy(np.float64).T

assert RESPONSE.shape == (4,218)
assert np.allclose(RESPONSE.sum(axis=1), 1, atol=1e-7, rtol=0)

def stretch(rgb, limits):
    lo, hi = limits
    return np.clip(
        (rgb.astype(np.float64)-lo)
        / np.maximum(hi-lo, 1e-12),
        0, 1
    ) ** 0.85

def display_rgb(msi, limits):
    rgb = np.moveaxis(msi[[2,1,0]], 0, -1)
    return stretch(rgb, limits)

def display_limits(msi):
    rgb = np.moveaxis(msi[[2,1,0]], 0, -1)
    return (
        np.percentile(rgb, 2, axis=(0,1)),
        np.percentile(rgb, 98, axis=(0,1))
    )

def load_patch(pid):
    path = S30 / "Paired_Observations" / f"{pid}.npz"
    if not path.is_file():
        raise FileNotFoundError(path)

    with np.load(path, allow_pickle=False) as z:
        ref = z["hrmsi"].copy()
        shifted = z["shifted_msi"].copy()
        offsets = z["shift_offsets_dy_dx"].copy()
        bandids = z["retained_band_ids"].copy()

    assert ref.shape == (4,96,96)
    assert shifted.shape == (17,4,96,96)
    assert offsets.shape == (17,2)
    assert np.array_equal(bandids.astype(int), retained_ids)

    aligned_indices = np.flatnonzero(
        np.all(np.isclose(offsets, 0, atol=1e-10), axis=1)
    )
    if len(aligned_indices) != 1:
        raise RuntimeError("Aligned observation ambiguous")
    if not np.array_equal(shifted[aligned_indices[0]], ref):
        raise RuntimeError("Aligned MSI does not match reference")
    return ref, shifted, offsets

def get_index(offsets, shift=(1.5, 0.0)):
    indexes = np.flatnonzero(
        np.all(np.isclose(offsets, shift, atol=1e-8), axis=1)
    )
    if len(indexes) != 1:
        raise RuntimeError("Chosen frozen shift not found")
    return int(indexes[0])

def get_estimate(pid, j, method):
    q = combined[
        (combined["patch_id"] == pid)
        & (combined["condition_index"] == j)
        & (combined["method"] == method)
    ]
    if len(q) != 1:
        raise RuntimeError("Missing saved displacement")
    row = q.iloc[0]
    return (
        float(row["estimated_dy"]),
        float(row["estimated_dx"]),
        float(row["epe"])
    )

def warp_for_display(observed, dy, dx):
    # Display only: inverse estimated translation.
    return ndi_shift(
        observed.astype(np.float32),
        shift=(0,-dy,-dx),
        order=1,
        mode="nearest",
        prefilter=False
    )

def gray_from_msi(msi):
    # Normalized visible-band intensity used only for overlays.
    return np.mean(
        msi[[0,1,2]].astype(np.float64),
        axis=0
    )

def edge_strength(msi):
    gray = gray_from_msi(msi)
    gy = sobel(gray, axis=0)
    gx = sobel(gray, axis=1)
    return np.hypot(gy, gx)

def binary_edges(msi, threshold):
    return edge_strength(msi) >= threshold

def overlay_edges(reference, comparison, threshold):
    # Reference edges: magenta; comparison edges: cyan.
    # White pixels indicate overlap of edge responses.
    r = binary_edges(reference, threshold)
    c = binary_edges(comparison, threshold)

    out = np.zeros((*r.shape, 3), dtype=float) + 0.06
    out[r] = (1.0, 0.28, 0.63)
    out[c] = (0.15, 0.85, 0.92)
    out[r & c] = (1.0, 1.0, 1.0)
    return out

# =============================================================================
# C — F02: FULL IMAGE, MAGNIFIED DETAIL, AND EDGE OVERLAY
# =============================================================================

print("\n45B — Building improved F02")

# Select patches by position, not by model performance.
EXAMPLES = [PATCH_IDS[0], PATCH_IDS[-1]]
SHIFT = (1.5, 0.0)

# The same inset region is used for all methods on each patch.
CROP_Y0, CROP_Y1 = 25, 65
CROP_X0, CROP_X1 = 25, 65

DISPLAY_METHODS = [
    (None, "Aligned reference"),
    ("shifted", "Displaced MSI"),
    (NCC, "Frozen NCC"),
    (NCC_DIAG, "NCC, bias omitted*"),
    (PHASE, "Frozen phase"),
    (PHASE_DIAG, "Phase, bias omitted*")
]

audit_rows = []

# Six columns and three rows per example:
# row 1: true-color full patch
# row 2: 40x40 central zoom
# row 3: magnified edge overlay.
# Two separate figures, assembled in one 2-page conceptual layout
# by stacking two per-patch blocks.

fig = plt.figure(figsize=(17, 13.1), constrained_layout=True)
outer = fig.add_gridspec(
    2, 1,
    height_ratios=[1,1],
    hspace=0.10
)

for block_index, pid in enumerate(EXAMPLES):
    reference, shifted, offsets = load_patch(pid)
    j = get_index(offsets, SHIFT)
    observed = shifted[j]
    limits = display_limits(reference)

    reference_edges = edge_strength(reference)
    # Fixed per-patch threshold determined solely from reference.
    EDGE_THRESHOLD = float(np.percentile(
        reference_edges[8:-8,8:-8], 85
    ))

    images = [reference, observed]
    epe_values = [None, float(np.hypot(*SHIFT))]

    for method, label in DISPLAY_METHODS[2:]:
        dy, dx, epe = get_estimate(pid,j,method)
        images.append(warp_for_display(observed,dy,dx))
        epe_values.append(epe)

        audit_rows.append({
            "patch_id": pid,
            "condition_index": j,
            "method": method,
            "estimated_dy": dy,
            "estimated_dx": dx,
            "registration_epe": epe,
            "edge_threshold_reference_85pct": EDGE_THRESHOLD
        })

    block_grid = outer[block_index].subgridspec(
        3, 6,
        height_ratios=[1.0, 0.88, 0.88],
        hspace=0.06, wspace=0.04
    )

    for col, ((method, label), image) in enumerate(
        zip(DISPLAY_METHODS, images)
    ):
        for row in range(3):
            ax = fig.add_subplot(block_grid[row,col])

            if row == 0:
                shown = display_rgb(image, limits)
                ax.imshow(shown, interpolation="nearest")
                ax.add_patch(Rectangle(
                    (CROP_X0-0.5, CROP_Y0-0.5),
                    CROP_X1-CROP_X0,
                    CROP_Y1-CROP_Y0,
                    fill=False,
                    edgecolor="#F6D55C",
                    linewidth=1.2
                ))

            elif row == 1:
                shown = display_rgb(image, limits)
                ax.imshow(
                    shown[CROP_Y0:CROP_Y1,CROP_X0:CROP_X1],
                    interpolation="nearest"
                )

            else:
                # The first column is reference edge map.
                if col == 0:
                    edge = binary_edges(reference, EDGE_THRESHOLD)
                    edge = edge[CROP_Y0:CROP_Y1,CROP_X0:CROP_X1]
                    ax.imshow(edge, cmap="gray", vmin=0, vmax=1)
                else:
                    overlay = overlay_edges(
                        reference, image, EDGE_THRESHOLD
                    )
                    ax.imshow(
                        overlay[CROP_Y0:CROP_Y1,CROP_X0:CROP_X1],
                        interpolation="nearest"
                    )

            ax.set_xticks([])
            ax.set_yticks([])

            for spine in ax.spines.values():
                spine.set_linewidth(0.55)
                spine.set_color("#777777")

            if row == 0:
                ax.set_title(
                    label, fontsize=8.5,
                    pad=7
                )
                if epe_values[col] is not None:
                    ax.text(
                        0.025, 0.035,
                        f"EPE {epe_values[col]:.2f} px",
                        transform=ax.transAxes,
                        color="white",
                        fontsize=7.5,
                        bbox=dict(
                            facecolor="black",
                            alpha=0.72,
                            edgecolor="none",
                            pad=1.6
                        )
                    )

            if col == 0:
                row_labels = [
                    f"{pid}\nFull patch",
                    "40×40 detail",
                    "Reference edges" if row == 2 else ""
                ]
                ax.set_ylabel(
                    row_labels[row],
                    fontsize=8
                )

fig.text(
    0.5, 0.01,
    "Edge overlay: magenta = reference, cyan = comparison, "
    "white = overlap. * Post hoc diagnostic; warping shown for visualization.",
    ha="center", va="bottom", fontsize=9
)

save_fig(
    fig,
    "F02_Refined_Registration_And_Edge_Overlays"
)

pd.DataFrame(audit_rows).to_csv(
    OUT / "Step45_Figure_Data_Audit.csv",
    index=False
)

# =============================================================================
# D — LOAD ACTUAL GEOGRAPHIC ENMAP IMAGERY
# =============================================================================

print("\n45C — Loading actual EnMAP scene for F04")

EXPECTED_TIF_SHA = (
    "4ba14fd27ac80c9169f3d733010dbdaa986584b3a062dc497578c3e114733329"
)

if file_sha(TIF) != EXPECTED_TIF_SHA:
    raise RuntimeError("Source EnMAP TIFF checksum mismatch")

with rasterio.open(TIF) as src:
    assert (src.count, src.height, src.width) == (224,200,330)
    assert src.crs is not None
    CRS = str(src.crs)
    assert CRS == "EPSG:32633"
    TRANSFORM = src.transform
    BOUNDS = src.bounds

    if not (
        np.isclose(TRANSFORM.b, 0)
        and np.isclose(TRANSFORM.d, 0)
    ):
        raise RuntimeError("Unexpected rotated image grid")

    # Rasterio indexes must be a plain Python list.
    CUBE = src.read(
        indexes=[int(i) for i in retained_ids]
    ).astype(np.float32)

assert CUBE.shape == (218,200,330)

scene_msi = np.einsum(
    "bc,chw->bhw",
    RESPONSE,
    CUBE.astype(np.float64),
    optimize=True
)

scene_rgb = np.moveaxis(
    scene_msi[[2,1,0]], 0, -1
)

scene_limits = (
    np.percentile(scene_rgb,2,axis=(0,1)),
    np.percentile(scene_rgb,98,axis=(0,1))
)

SCENE_DISPLAY = stretch(scene_rgb,scene_limits)

XMIN, XMAX = BOUNDS.left/1000, BOUNDS.right/1000
YMIN, YMAX = BOUNDS.bottom/1000, BOUNDS.top/1000

def patch_box(pid):
    row,col = patch_row_col(pid)
    x0,y0 = TRANSFORM*(col,row)
    x1,y1 = TRANSFORM*(col+96,row+96)
    return (
        min(x0,x1)/1000,
        max(x0,x1)/1000,
        min(y0,y1)/1000,
        max(y0,y1)/1000
    )

# =============================================================================
# E — COMPACT F04 PATCH-LEVEL ERROR
# =============================================================================

plot_methods = [
    (NCC, "Frozen NCC"),
    (PHASE, "Frozen bounded phase"),
    (NCC_DIAG, "NCC, bias omitted*"),
    (PHASE_DIAG, "Phase, bias omitted*")
]

patch_means = (
    combined[
        combined["method"].isin([m for m,_ in plot_methods])
    ]
    .groupby(["method","patch_id"],as_index=False)
    .agg(
        cases=("epe","size"),
        mean_epe=("epe","mean")
    )
)

if not (patch_means["cases"] == 17).all():
    raise RuntimeError("Expected 17 conditions for every patch")

patch_means.to_csv(
    OUT / "Step45_Patch_EPE.csv",
    index=False
)

lookup = {}
for method,label in plot_methods:
    sub = patch_means[patch_means["method"] == method]
    lookup[method] = dict(zip(
        sub["patch_id"],sub["mean_epe"].astype(float)
    ))

cmap = plt.get_cmap("YlOrRd")
norm = Normalize(
    vmin=0,
    vmax=max(1.7, float(patch_means["mean_epe"].max()))
)

fig, axes = plt.subplots(
    2,2,
    figsize=(12.6,8.4),
    gridspec_kw={
        "wspace":0.08,
        "hspace":0.16
    }
)

for k,(ax,(method,label)) in enumerate(
    zip(axes.flat,plot_methods)
):
    ax.imshow(
        SCENE_DISPLAY,
        extent=[XMIN,XMAX,YMIN,YMAX],
        origin="upper",
        interpolation="nearest",
        alpha=0.91
    )

    for i,pid in enumerate(PATCH_IDS,1):
        x0,x1,y0,y1 = patch_box(pid)
        val = lookup[method][pid]

        ax.add_patch(Rectangle(
            (x0,y0),x1-x0,y1-y0,
            facecolor=cmap(norm(val)),
            edgecolor="white",
            linewidth=1.25,
            alpha=0.64,
            zorder=4
        ))
        ax.text(
            (x0+x1)/2,(y0+y1)/2,
            f"P{i}\n{val:.2f}",
            ha="center",va="center",
            fontsize=8.5,fontweight="bold",
            bbox=dict(
                facecolor="white",
                edgecolor="none",
                alpha=0.88,
                pad=1.3
            ),
            zorder=7
        )

    ax.set_aspect("equal")
    ax.set_xlim(XMIN,XMAX)
    ax.set_ylim(YMIN,YMAX)
    ax.set_title(
        f"({chr(97+k)}) {label}",
        loc="left",
        fontsize=11,
        pad=6
    )

    ax.tick_params(
        direction="out",
        labelsize=7,
        top=True,right=True
    )

    if k in [2,3]:
        ax.set_xlabel("Easting (km)")
    else:
        ax.tick_params(labelbottom=False)

    if k in [0,2]:
        ax.set_ylabel("Northing (km)")
    else:
        ax.tick_params(labelleft=False)

fig.subplots_adjust(
    left=0.07,
    right=0.88,
    bottom=0.08,
    top=0.94,
    hspace=0.15,
    wspace=0.08
)

cax = fig.add_axes([0.90,0.22,0.022,0.55])

sm = ScalarMappable(norm=norm,cmap=cmap)
sm.set_array([])

cb = fig.colorbar(sm,cax=cax)
cb.set_label(
    "Patch mean endpoint error (pixels)",
    fontsize=9
)

save_fig(
    fig,
    "F04_Refined_Compact_Spatial_EPE"
)

# =============================================================================
# F — CAPTIONS AND FINAL REPORT
# =============================================================================

captions = """
F02. Registration and edge-overlap examples for two fixed
Potsdam patches under a synthetic 1.5-pixel vertical translation.
For each patch, full RGB composites, 40 × 40-pixel central
details and reference/comparison edge overlays are displayed.
All corrected images are visualized using inverse shifts of
the saved Step41C/Step42 displacement estimates. Edge thresholds
are computed from each aligned reference image and held
fixed across methods. Magenta indicates reference edges, cyan
indicates comparison edges, and white marks overlapping edge
responses. Endpoint errors derive from the saved registration
evaluation rather than newly computed image differences.
The bias-omission variants are post hoc diagnostics.

F04. Compact spatial summary of mean registration endpoint error
across five non-overlapping Potsdam patches. Each patch value
summarizes 17 synthetic displacement conditions from a single
EnMAP acquisition. The upper panels show the frozen original
MDAS NCC and bounded-phase transfer results; the lower panels
show exploratory bias-omission counterfactuals. All four maps
use the same projected coordinates and shared error scale.
The colour overlay summarizes patch-level errors rather than
pixelwise registration fields.
"""

(OUT / "Step45_Figure_Captions.txt").write_text(
    captions.strip(),encoding="utf-8"
)

report = {
    "status":"REFINED_SPATIAL_FIGURES_GENERATED",
    "figure_count":2,
    "png_dpi":900,
    "pdf_vector_exports":2,
    "original_enmap_used":True,
    "source_crs":CRS,
    "source_acquisitions":1,
    "patch_count":5,
    "synthetic_conditions_per_patch":17,
    "registration_estimators_rerun":False,
    "registration_parameters_modified":False,
    "image_warping_for_visualization_only":True,
    "posthoc_diagnostics_explicit":True,
    "source_sha256":{
        "enmap":file_sha(TIF),
        "protocol":file_sha(PROTOCOL_PATH),
        "response":file_sha(RESPONSE_PATH),
        "step41c":file_sha(FROZEN_CSV),
        "step42":file_sha(DIAGNOSTIC_CSV)
    },
    "output_directory":str(OUT),
    "elapsed_seconds":round(time.monotonic()-START,2)
}

save_json(OUT/"Step45_Final_Report.json",report)

print("\n"+"="*88)
print("STEP 45 — FINAL REPORT")
print("="*88)
print(json.dumps(report,indent=2))
print("\nSTEP 45 COMPLETE")
print("Previous figure files and experimental outputs unchanged.")


In [ ]:

# =============================================================================
# STEP 46 — FINAL MDAS MANUSCRIPT FIGURE AND TABLE PACKAGE
# =============================================================================
#
# PURPOSE
#   Consolidate all verified NEW manuscript figures and tables.
#   Correct the Step45 F02 layout by removing overlapping footer text.
#   Separate frozen original registration from post hoc diagnostics.
#   Prepare a single ZIP package for manuscript revision in ChatGPT.
#
# FIGURES
#   N01  External mean endpoint error
#   N02  Aligned displacement diagnostics
#   N03  Actual Potsdam EnMAP scene and sampling patches
#   N04  Registration images + edge overlays, patch P1
#   N05  Registration images + edge overlays, patch P5
#   N06  Four-band MSI difference maps
#   N07  Spatial patch-level endpoint-error maps
#
# TABLES
#   External performance
#   Patch-level results
#   Post hoc bias attribution
#   Historical reproduction
#   Registration comparison
#   Additional saved numerical summaries
#
# OUTPUT
#   Manuscript_Figures/          PNG + vector PDF
#   Manuscript_Tables/           CSV + readable Markdown tables
#   Figure_Captions.md
#   Figure_and_Table_Inventory.csv
#   Manuscript_Insertion_Guide.md
#   Source_Provenance.json
#   Complete ZIP package
#
# Original experiments and figures are never overwritten.
# =============================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
from PIL import Image
from scipy.ndimage import shift as ndi_shift, sobel

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hashlib
import json
import shutil
import zipfile
import re
import time

START = time.monotonic()

# =============================================================================
# 46A — PROJECT AND SOURCE PATHS
# =============================================================================

PROJECT = Path(
    "/content/drive/MyDrive/"
    "MDAS_Registration_Aware_Revision_2026"
)

S30 = PROJECT / "Step30_Frozen_Potsdam_Benchmark"
S40 = PROJECT / "Step40_Reproduction_Discrepancy_Audit"
S41 = PROJECT / "Step41C_Frozen_Potsdam_Transfer"
S42 = PROJECT / "Step42_External_Calibration_Transfer_Audit"
S43 = PROJECT / "Step43_External_Manuscript_Results"
S44 = PROJECT / "Step44C_Final_Verified_Spatial_Figures"
S45 = PROJECT / "Step45_Refined_Spatial_Manuscript_Figures"

# Each execution receives a new output folder.
# No previous research products are overwritten.

RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S"
)

OUT = (
    PROJECT
    / f"Step46_Final_Manuscript_Package_{RUN_ID}"
)

FIG_DIR = OUT / "Manuscript_Figures"
TABLE_DIR = OUT / "Manuscript_Tables"

FIG_DIR.mkdir(parents=True, exist_ok=False)
TABLE_DIR.mkdir(parents=True, exist_ok=False)

print("=" * 90)
print("STEP 46 — FINAL MANUSCRIPT FIGURE AND TABLE PACKAGE")
print("=" * 90)
print("Output:", OUT)

# =============================================================================
# 46B — HELPERS
# =============================================================================

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)
    return h.hexdigest()


def load_json(path):
    return json.loads(
        path.read_text(encoding="utf-8")
    )


def save_json(path, data):
    path.write_text(
        json.dumps(
            data,
            indent=2,
            allow_nan=False,
            default=str
        ),
        encoding="utf-8"
    )


def require_file(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


def copy_verified(source, destination):
    require_file(source)

    if destination.exists():
        raise FileExistsError(destination)

    shutil.copy2(source, destination)

    if sha256(source) != sha256(destination):
        raise RuntimeError(
            "File copy checksum mismatch: "
            + str(source)
        )

    return destination


def save_figure(fig, stem):

    png = FIG_DIR / f"{stem}.png"
    pdf = FIG_DIR / f"{stem}.pdf"

    if png.exists() or pdf.exists():
        raise FileExistsError(stem)

    fig.savefig(
        png,
        dpi=900,
        bbox_inches="tight",
        pad_inches=0.10,
        facecolor="white"
    )

    fig.savefig(
        pdf,
        bbox_inches="tight",
        pad_inches=0.10,
        facecolor="white"
    )

    plt.show()
    plt.close(fig)

    print("Generated:", stem)


def save_table(df, stem):
    path = TABLE_DIR / f"{stem}.csv"
    if path.exists():
        raise FileExistsError(path)
    df.to_csv(path, index=False)
    return path


plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

# =============================================================================
# 46C — VERIFY THE EXPERIMENTS
# =============================================================================

report_paths = {
    "step40": S40 / "Step40_Final_Report.json",
    "step41c": S41 / "Step41C_Final_Report.json",
    "step42": S42 / "Step42_Final_Report.json",
    "step43": S43 / "Step43_Final_Report.json",
    "step44c": S44 / "Step44C_Final_Report.json",
    "step45": S45 / "Step45_Final_Report.json",
}

for path in report_paths.values():
    require_file(path)

reports = {
    name: load_json(path)
    for name, path in report_paths.items()
}

expected_status = {
    "step40": "NUMERICAL_REPRODUCTION_CONSISTENT",
    "step41c": "EXTERNAL_FROZEN_REGISTRATION_EVALUATION_COMPLETE",
    "step42": "POST_HOC_CALIBRATION_TRANSFER_AUDIT_COMPLETED",
    "step43": "EXTERNAL_MANUSCRIPT_RESULTS_PACKAGE_COMPLETE",
    "step44c": "SPATIAL_FIGURES_COMPLETE",
    "step45": "REFINED_SPATIAL_FIGURES_GENERATED",
}

for step, status in expected_status.items():

    actual = reports[step].get("status")

    if actual != status:
        raise RuntimeError(
            f"{step} status mismatch: {actual}"
        )

FROZEN_CSV = require_file(
    S41 / "Step41C_Registration_Results.csv"
)

DIAGNOSTIC_CSV = require_file(
    S42 / "Step42_Bias_Attribution_Cases.csv"
)

if reports["step42"]["source_step41c_results_sha256"] != sha256(
    FROZEN_CSV
):
    raise RuntimeError(
        "Step42 provenance does not match Step41C."
    )

frozen = pd.read_csv(FROZEN_CSV)
combined = pd.read_csv(DIAGNOSTIC_CSV)

assert len(frozen) == 255
assert len(combined) == 425

NCC = "frozen_ncc_step13"
PHASE = "frozen_phase_step20_size_adapted"
IDENTITY = "identity"

NCC_DIAG = "ncc_prebias_same_gate_diagnostic"
PHASE_DIAG = "phase_prebias_same_gate_diagnostic"

METHODS = [
    NCC, PHASE, IDENTITY,
    NCC_DIAG, PHASE_DIAG
]

assert set(combined["method"]) == set(METHODS)

KEY = ["patch_id", "condition_index"]

for frame in [frozen, combined]:

    if frame.duplicated(KEY + ["method"]).any():
        raise RuntimeError(
            "Duplicated registration case keys"
        )

    for col in [
        "true_dy",
        "true_dx",
        "estimated_dy",
        "estimated_dx",
        "epe"
    ]:
        frame[col] = pd.to_numeric(
            frame[col],
            errors="raise"
        )

    if not np.isfinite(
        frame[
            [
                "true_dy",
                "true_dx",
                "estimated_dy",
                "estimated_dx",
                "epe"
            ]
        ].to_numpy(float)
    ).all():
        raise RuntimeError(
            "Nonfinite experimental results"
        )

if not (combined.groupby("method").size() == 85).all():
    raise RuntimeError(
        "Expected 85 cases per method."
    )

# Reconcile frozen records against Step42 copies,
# using matching case keys and CSV-safe precision.

comparison_tolerance = 1e-9
maximum_cross_step_difference = 0.0

for method in [NCC, PHASE, IDENTITY]:

    a = frozen[
        frozen["method"] == method
    ][KEY + [
        "true_dy", "true_dx",
        "estimated_dy", "estimated_dx", "epe"
    ]]

    b = combined[
        combined["method"] == method
    ][KEY + [
        "true_dy", "true_dx",
        "estimated_dy", "estimated_dx", "epe"
    ]]

    joined = a.merge(
        b,
        on=KEY,
        how="outer",
        validate="one_to_one",
        indicator=True,
        suffixes=("_41c", "_42")
    )

    if (
        len(joined) != 85
        or not joined["_merge"].eq("both").all()
    ):
        raise RuntimeError(
            f"Case reconciliation failed: {method}"
        )

    for field in [
        "true_dy", "true_dx",
        "estimated_dy", "estimated_dx", "epe"
    ]:

        a_values = joined[
            field + "_41c"
        ].to_numpy(float)

        b_values = joined[
            field + "_42"
        ].to_numpy(float)

        diff = float(
            np.max(np.abs(a_values - b_values))
        )

        maximum_cross_step_difference = max(
            maximum_cross_step_difference,
            diff
        )

        if diff > comparison_tolerance:
            raise RuntimeError(
                f"Frozen data discrepancy: "
                f"{method}/{field}: {diff}"
            )

print("\n46C — ALL SOURCE RESULTS VERIFIED")
print("Frozen records:", len(frozen))
print("Post hoc records:", len(combined)-len(frozen))
print(
    "Maximum cross-step numerical difference:",
    maximum_cross_step_difference
)

# =============================================================================
# 46D — COLLECT THE BEST PREVIOUS FIGURES
# =============================================================================

# We retain existing VERIFIED images without rerendering them.
#
# N01/N02: Step43 statistical figures
# N03/N06: Step44C real EnMAP figures
# N07: Step45 refined spatial EPE figure
#
# N04/N05 are rendered afresh below.

existing_figure_sources = {
    "N01_External_Mean_EPE": (
        S43 / "Step43_Figure_External_EPE"
    ),
    "N02_Aligned_Residuals": (
        S43 / "Step43_Figure_Aligned_Residuals"
    ),
    "N03_EnMAP_Scene_and_Patches": (
        S44 / "F01_EnMAP_Scene_Patches"
    ),
    "N06_Four_Band_Difference_Maps": (
        S44 / "F03_Four_Band_Difference_Maps"
    ),
    "N07_Spatial_Patch_Mean_EPE": (
        S45 / "F04_Refined_Compact_Spatial_EPE"
    ),
}

figure_inventory = []

for new_stem, original_stem in existing_figure_sources.items():

    source_png = original_stem.with_suffix(".png")
    source_pdf = original_stem.with_suffix(".pdf")

    copied_png = copy_verified(
        source_png,
        FIG_DIR / f"{new_stem}.png"
    )

    copied_pdf = copy_verified(
        source_pdf,
        FIG_DIR / f"{new_stem}.pdf"
    )

    with Image.open(copied_png) as im:
        size = im.size

    figure_inventory.append({
        "figure_id": new_stem.split("_")[0],
        "filename_stem": new_stem,
        "source": str(original_stem),
        "png_width_px": int(size[0]),
        "png_height_px": int(size[1]),
        "png_sha256": sha256(copied_png),
        "pdf_sha256": sha256(copied_pdf),
        "evidence_status": (
            "PRIMARY"
            if new_stem.startswith(("N01", "N03", "N06"))
            else "MIXED_PRIMARY_AND_EXPLORATORY"
        )
    })

    print("Copied:", new_stem)

# =============================================================================
# 46E — GENERATE CORRECTED REGISTRATION FIGURES N04 / N05
# =============================================================================

# This replaces Step45 F02 with two readable figures.
# No overlapping footer text.
# One spatial patch per figure.
#
# Rows:
#   1. Full 96x96 image
#   2. Enlarged 40x40 detail
#   3. Reference-versus-correction edge overlay
#
# Columns:
#   Reference, displaced, frozen NCC, NCC diagnostic,
#   frozen phase, phase diagnostic.

PROTOCOL_PATH = require_file(
    S30 / "Step30_Frozen_Forward_Model.json"
)

protocol = load_json(PROTOCOL_PATH)

assert protocol["source_scene_count"] == 1
assert protocol["retained_band_count"] == 218

def patch_coordinates(pid):
    m = re.fullmatch(
        r"potsdam_r(\d+)_c(\d+)",
        pid
    )
    if m is None:
        raise ValueError(
            f"Unexpected patch ID: {pid}"
        )
    return int(m.group(1)), int(m.group(2))

PATCH_IDS = sorted(
    protocol["selected_patch_ids"],
    key=patch_coordinates
)

assert len(PATCH_IDS) == 5

retained_band_ids = np.asarray(
    protocol["retained_band_ids_1_based"],
    dtype=int
)

def load_patch(pid):

    path = require_file(
        S30 / "Paired_Observations"
        / f"{pid}.npz"
    )

    with np.load(
        path, allow_pickle=False
    ) as z:

        reference = z["hrmsi"].copy()
        shifted = z["shifted_msi"].copy()
        offsets = z[
            "shift_offsets_dy_dx"
        ].copy()

        band_ids = z[
            "retained_band_ids"
        ].copy()

    if reference.shape != (4,96,96):
        raise RuntimeError("Unexpected reference MSI shape")

    if shifted.shape != (17,4,96,96):
        raise RuntimeError("Unexpected displaced MSI shape")

    if offsets.shape != (17,2):
        raise RuntimeError("Unexpected shift-array shape")

    if not np.array_equal(
        band_ids.astype(int),
        retained_band_ids
    ):
        raise RuntimeError("Frozen spectral-band mismatch")

    zero = np.flatnonzero(
        np.all(
            np.isclose(
                offsets, [0,0],
                atol=1e-10
            ),
            axis=1
        )
    )

    if (
        len(zero) != 1
        or not np.array_equal(
            shifted[zero[0]], reference
        )
    ):
        raise RuntimeError(
            "Aligned observation verification failed"
        )

    return reference, shifted, offsets


def find_condition(offsets, dy=1.5, dx=0.0):

    matches = np.flatnonzero(
        np.all(
            np.isclose(
                offsets,
                [dy,dx],
                atol=1e-8
            ),
            axis=1
        )
    )

    if len(matches) != 1:
        raise RuntimeError(
            "Chosen synthetic shift is missing"
        )

    return int(matches[0])


def get_estimate(pid, index, method):

    sub = combined[
        (combined["patch_id"] == pid)
        & (combined["condition_index"] == index)
        & (combined["method"] == method)
    ]

    if len(sub) != 1:
        raise RuntimeError(
            "Missing saved registration estimate"
        )

    row = sub.iloc[0]

    return (
        float(row["estimated_dy"]),
        float(row["estimated_dx"]),
        float(row["epe"])
    )


def visual_warp(msi, dy, dx):

    return ndi_shift(
        msi.astype(np.float32),
        shift=(0,-dy,-dx),
        order=1,
        mode="nearest",
        prefilter=False
    )


def rgb_limits(reference):

    rgb = np.moveaxis(
        reference[[2,1,0]],
        0,-1
    )

    return (
        np.percentile(
            rgb,2,axis=(0,1)
        ),
        np.percentile(
            rgb,98,axis=(0,1)
        )
    )


def render_rgb(msi, limits):

    rgb = np.moveaxis(
        msi[[2,1,0]],0,-1
    ).astype(np.float64)

    low, high = limits

    return np.clip(
        (rgb-low)
        /np.maximum(high-low,1e-12),
        0,1
    )**0.85


def edge_magnitude(msi):

    visible = np.mean(
        msi[[0,1,2]].astype(np.float64),
        axis=0
    )

    return np.hypot(
        sobel(visible,axis=0),
        sobel(visible,axis=1)
    )


def edge_overlay(reference, comparison, threshold):

    ref = edge_magnitude(reference) >= threshold
    obs = edge_magnitude(comparison) >= threshold

    result = np.zeros(
        (96,96,3),
        dtype=np.float32
    ) + 0.055

    # Reference edges: magenta.
    result[ref] = [1.0,0.24,0.64]

    # Comparison edges: cyan.
    result[obs] = [0.12,0.84,0.92]

    # Common edges: white.
    result[ref & obs] = [1,1,1]

    return result


PLOT_COLUMNS = [
    (None,"Aligned reference"),
    ("shifted","Displaced MSI"),
    (NCC,"Frozen NCC"),
    (NCC_DIAG,"NCC, bias omitted*"),
    (PHASE,"Frozen phase"),
    (PHASE_DIAG,"Phase, bias omitted*")
]

CROP = (25,65,25,65)

def generate_registration_figure(pid, figure_id):

    reference, shifted, offsets = load_patch(pid)

    condition = find_condition(
        offsets,dy=1.5,dx=0.0
    )

    observed = shifted[condition]

    limits = rgb_limits(reference)

    threshold = float(
        np.percentile(
            edge_magnitude(reference)[8:-8,8:-8],
            85
        )
    )

    images = [reference, observed]
    errors = [None, 1.5]

    audit_records = []

    for method,label in PLOT_COLUMNS[2:]:

        dy,dx,epe = get_estimate(
            pid,condition,method
        )

        corrected = visual_warp(
            observed,dy,dx
        )

        images.append(corrected)
        errors.append(epe)

        audit_records.append({
            "figure":figure_id,
            "patch_id":pid,
            "method":method,
            "condition_index":condition,
            "true_dy":1.5,
            "true_dx":0.0,
            "estimated_dy":dy,
            "estimated_dx":dx,
            "saved_epe":epe,
            "edge_threshold":threshold
        })

    fig, axes = plt.subplots(
        3,6,
        figsize=(15.3,7.5),
        gridspec_kw={
            "height_ratios":[1,1,1],
            "hspace":0.12,
            "wspace":0.06
        }
    )

    for col, (
        (method,label), image
    ) in enumerate(
        zip(PLOT_COLUMNS,images)
    ):

        rgb = render_rgb(image,limits)

        y0,y1,x0,x1 = CROP

        for row in range(3):

            ax = axes[row,col]

            if row == 0:

                ax.imshow(
                    rgb,
                    interpolation="nearest"
                )

                ax.add_patch(
                    plt.Rectangle(
                        (x0-0.5,y0-0.5),
                        x1-x0,
                        y1-y0,
                        edgecolor="#F8DA64",
                        facecolor="none",
                        lw=1.2
                    )
                )

                if errors[col] is not None:

                    ax.text(
                        0.035,0.05,
                        f"EPE {errors[col]:.2f} px",
                        transform=ax.transAxes,
                        fontsize=8,
                        color="white",
                        bbox=dict(
                            facecolor="black",
                            alpha=0.75,
                            edgecolor="none",
                            pad=2
                        )
                    )

                ax.set_title(
                    f"({chr(97+col)}) {label}",
                    fontsize=9.2,
                    pad=8
                )

            elif row == 1:

                ax.imshow(
                    rgb[y0:y1,x0:x1],
                    interpolation="nearest"
                )

            else:

                if col == 0:

                    ref_edge = (
                        edge_magnitude(reference)
                        >=threshold
                    )

                    ax.imshow(
                        ref_edge[
                            y0:y1,x0:x1
                        ],
                        cmap="gray",
                        vmin=0,vmax=1,
                        interpolation="nearest"
                    )

                else:

                    overlay = edge_overlay(
                        reference,
                        image,
                        threshold
                    )

                    ax.imshow(
                        overlay[
                            y0:y1,x0:x1
                        ],
                        interpolation="nearest"
                    )

            ax.set_xticks([])
            ax.set_yticks([])

            for spine in ax.spines.values():
                spine.set_color("#777777")
                spine.set_linewidth(0.6)

    axes[0,0].set_ylabel(
        "Full image\n96 × 96",
        fontsize=9
    )

    axes[1,0].set_ylabel(
        "Magnified detail\n40 × 40",
        fontsize=9
    )

    axes[2,0].set_ylabel(
        "Edge overlap\n40 × 40",
        fontsize=9
    )

    # No bottom footer is placed in the figure.
    # All explanations belong in the manuscript caption.

    fig.subplots_adjust(
        left=0.055,
        right=0.995,
        top=0.93,
        bottom=0.045,
        hspace=0.16,
        wspace=0.065
    )

    stem = (
        f"{figure_id}_Registration_"
        + pid
    )

    save_figure(fig,stem)

    return stem, audit_records


# The two patches are selected by spatial position,
# not by registration accuracy.

NEW_FIGURES = []
example_audit = []

for pid, figure_id in [
    (PATCH_IDS[0],"N04"),
    (PATCH_IDS[-1],"N05")
]:

    stem, rows = generate_registration_figure(
        pid,figure_id
    )

    NEW_FIGURES.append(stem)
    example_audit.extend(rows)

    png = FIG_DIR/f"{stem}.png"
    pdf = FIG_DIR/f"{stem}.pdf"

    with Image.open(png) as im:
        w,h = im.size

    figure_inventory.append({
        "figure_id":figure_id,
        "filename_stem":stem,
        "source":(
            "Step30 imagery + "
            "Step41C/42 saved estimates"
        ),
        "png_width_px":w,
        "png_height_px":h,
        "png_sha256":sha256(png),
        "pdf_sha256":sha256(pdf),
        "evidence_status":(
            "MIXED_PRIMARY_AND_EXPLORATORY"
        )
    })

save_table(
    pd.DataFrame(example_audit),
    "T_Registration_Example_Audit"
)

# =============================================================================
# 46F — COLLECT THE VERIFIED NUMERICAL TABLES
# =============================================================================

table_sources = {
    "T01_External_Performance":
        S43 / "Step43_Table_External_Performance.csv",

    "T02_Patch_Level":
        S43 / "Step43_Table_Patch_Level.csv",

    "T03_Bias_Attribution":
        S43 / "Step43_Table_Bias_Attribution.csv",

    "T04_Historical_Reproduction":
        S40 / "Step40_Reproduction_Summary.csv",

    "T05_Phase_Tolerance_Sensitivity":
        S40 / "Step40_Tolerance_Sensitivity.csv",

    "T06_Frozen_External_Registration":
        S41 / "Step41C_Method_Summary.csv",

    "T07_Frozen_Patch_Results":
        S41 / "Step41C_Patch_Summary.csv",

    "T08_Frozen_Paired_Comparison":
        S41 / "Step41C_Paired_Comparison.csv",

    "T09_Posthoc_Diagnostic_Summary":
        S42 / "Step42_Descriptive_Method_Summary.csv",

    "T10_Registration_Example_Values":
        S44 / "Step44C_Example_Values.csv",

    "T11_Patch_Footprints":
        S44 / "Step44C_Patch_Footprints.csv",
}

table_inventory = []

for stem, source in table_sources.items():

    require_file(source)

    target = copy_verified(
        source,
        TABLE_DIR / f"{stem}.csv"
    )

    frame = pd.read_csv(target)

    table_inventory.append({
        "table_id":stem.split("_")[0],
        "filename":target.name,
        "rows":len(frame),
        "columns":len(frame.columns),
        "source":str(source),
        "sha256":sha256(target)
    })

    print(
        "Table:",
        stem,
        f"({len(frame)} rows)"
    )

# =============================================================================
# 46G — VERIFIED MANUSCRIPT CAPTIONS
# =============================================================================

captions = """
# New MDAS manuscript figure captions

## N01 — External registration performance

Mean registration endpoint error for frozen MDAS NCC,
frozen bounded phase correlation, the no-correction
baseline, and exploratory post hoc bias-omission diagnostics
on the Potsdam synthetic HSI–MSI benchmark. Each method
was assessed under 85 synthetic translation conditions
across five nonoverlapping patches from one EnMAP
acquisition. The frozen methods constitute the primary
external transfer experiment. Bias-omission results were
examined after the frozen-transfer outcomes and therefore
do not represent independent validation.

## N02 — Aligned displacement diagnostics

Residual displacement estimates for truly aligned Potsdam
observations under the frozen MDAS registrations and the
post hoc bias-omission diagnostics. The correct displacement
is zero. All five aligned observations originate from one
external EnMAP acquisition. This figure is descriptive and
should not be interpreted as five independent external
validation scenes.

## N03 — Potsdam EnMAP scene and selected patches

Georeferenced representation of the Potsdam EnMAP scene
with five nonoverlapping 96 × 96-pixel evaluation windows.
Patch locations were derived from their saved raster
row-column identifiers and the source image geotransform.
The displayed composite uses the frozen four-channel
synthetic spectral-response weights. The source raster
uses EPSG:32633 projected coordinates.

## N04 — Registration examples, patch P1

Aligned reference, synthetically displaced MSI, frozen
MDAS NCC and bounded-phase corrections, and exploratory
bias-omission diagnostics for the first selected Potsdam
patch under a 1.5-pixel vertical translation. Full
96 × 96-pixel images, common 40 × 40-pixel details and
edge-overlap diagnostics are shown. Magenta denotes
reference edges, cyan denotes comparison edges and white
denotes overlapping edge detections. Registration
endpoint errors are taken from the saved numerical
evaluation; inverse resampling was performed solely for
visualization. Bias-omission corrections are post hoc
diagnostics.

## N05 — Registration examples, patch P5

Registration comparison for the fifth selected Potsdam
patch using the same observation protocol, display
conventions, 1.5-pixel translation, magnification and
edge-overlay method as N04. The example was selected
by spatial position rather than registration performance.
The bias-omission variants are exploratory.

## N06 — Four-band MSI difference maps

Pixelwise mean absolute difference between an aligned
synthetic four-channel MSI reference and the displaced
or visually corrected MSI in one Potsdam patch. All
panels share a common colour scale. A six-pixel image
border is excluded to reduce edge-resampling effects.
The differences describe synthetic multispectral image
signals, not fused hyperspectral reconstruction errors.
Bias-omission results are exploratory.

## N07 — Spatial patch-level registration errors

Spatial pattern of patch-level mean endpoint error for
frozen original NCC, frozen bounded phase correlation and
their exploratory bias-omission diagnostics. Each patch
value averages 17 synthetic translation conditions. The
four panels use the same projected scene extent and
colour scale. All five patches belong to a single EnMAP
acquisition, and the coloured regions are patch-level
summaries rather than pixelwise displacement fields.

# Notes for manuscript preparation

- Rename N01–N07 to the final sequential figure numbers only
  after reviewing the full manuscript.
- Distinguish the frozen external transfer from post hoc
  diagnostics in all captions, legends and discussions.
- Do not describe the Potsdam benchmark as five independent
  source scenes.
- Do not describe four-band signal-difference maps as
  hyperspectral fusion-error maps.
- Preserve the important original Augsburg fusion
  architecture and spatial reconstruction figures.
"""

(OUT/"Figure_Captions.md").write_text(
    captions.strip(),
    encoding="utf-8"
)

# =============================================================================
# 46H — FIGURE INVENTORY AND TABLE INVENTORY
# =============================================================================

figure_inventory_df = pd.DataFrame(
    figure_inventory
).sort_values("figure_id")

assert len(figure_inventory_df) == 7
assert figure_inventory_df["figure_id"].nunique() == 7

figure_inventory_df.to_csv(
    OUT/"Figure_and_Table_Inventory.csv",
    index=False
)

pd.DataFrame(table_inventory).to_csv(
    OUT/"Table_Inventory.csv",
    index=False
)

# Generate a single readable Markdown document
# with the core tables.

CORE_TABLES = [
    "T01_External_Performance",
    "T03_Bias_Attribution",
    "T04_Historical_Reproduction",
    "T06_Frozen_External_Registration"
]

sections = [
    "# New MDAS results tables",
    "",
    "All tables are derived from previously saved experiments.",
    "Repeated shift conditions are not independent source scenes.",
    ""
]

for stem in CORE_TABLES:

    table = pd.read_csv(
        TABLE_DIR/f"{stem}.csv"
    )

    sections.extend([
        f"## {stem}",
        "",
        table.round(5).to_markdown(index=False),
        ""
    ])

(OUT/"Core_Tables_For_Manuscript.md").write_text(
    "\n".join(sections),
    encoding="utf-8"
)

# =============================================================================
# 46I — MANUSCRIPT INSERTION GUIDE
# =============================================================================

guide = """
# MDAS manuscript insertion guide

This package contains new external registration and
reproducibility figures only. It does not replace the
original Augsburg manuscript figures.

## Suggested main manuscript figures

N03 — Place in Study Area / External Dataset subsection.
N04 — Place in external registration visual analysis.
N06 — Place with the synthetic MSI image-difference results.
N07 — Place with external calibration-transfer analysis.
N01 — Use if numerical performance comparisons need a
standalone figure rather than a table.

## Suggested supplementary figures

N02 — Aligned residual diagnostic plot.
N05 — Additional image registration example from a second patch.

## Suggested main manuscript tables

T01 — External registration performance (frozen methods and
post hoc diagnostics explicitly separated).
T03 — Post hoc calibration-bias attribution, labelled exploratory.
T04 — Historical estimator numerical reproducibility.

## Suggested supplementary tables

T02, T05, T07, T08, T09, T10, T11 — detailed patch,
method, tolerance and example information.

## Important limitations

- One independent external EnMAP acquisition.
- Five nonoverlapping source patches.
- Seventeen repeated synthetic shifts per patch.
- Synthetic MSI rather than independently acquired multispectral data.
- Frozen MDAS biases applied without external recalibration.
- Post hoc bias omission is descriptive, not independently validated.
- No Potsdam fusion checkpoint inference was performed.

## Rules for revising the existing manuscript

1. Review every original figure and table before removal.
2. Preserve the original architecture diagram unless it is
   genuinely redundant or scientifically incorrect.
3. Keep the strongest Augsburg fusion reconstruction panels.
4. Avoid presenting repeated plots of the same mean EPE values.
5. Cross-reference every new table and figure in the text.
6. Keep figure and table numbering sequential.
7. Separate Results from exploratory diagnostic interpretation.
8. Include relevant reproducibility and validation limitations.

Final decisions about figure removal, table replacement and
placement must be made after reading the current full manuscript.
"""

(OUT/"Manuscript_Insertion_Guide.md").write_text(
    guide.strip(),
    encoding="utf-8"
)

# =============================================================================
# 46J — GENERATE A LIGHTWEIGHT VISUAL CONTACT SHEET
# =============================================================================

# Downsample only for the contact sheet.
# Original 900-dpi images remain unchanged.

ordered_stems = (
    figure_inventory_df["filename_stem"].tolist()
)

cols = 2
rows = int(np.ceil(len(ordered_stems)/cols))

thumb_w = 850
thumb_h = 520

sheet = Image.new(
    "RGB",
    (cols*thumb_w, rows*thumb_h),
    color="white"
)

from PIL import ImageDraw

draw = ImageDraw.Draw(sheet)

for idx, stem in enumerate(ordered_stems):

    source = FIG_DIR/f"{stem}.png"

    with Image.open(source) as im:
        thumb = im.convert("RGB")
        thumb.thumbnail(
            (thumb_w-40,thumb_h-60)
        )

    x0 = (idx%cols)*thumb_w
    y0 = (idx//cols)*thumb_h

    x = x0+(thumb_w-thumb.width)//2
    y = y0+38

    sheet.paste(thumb,(x,y))

    draw.text(
        (x0+20,y0+12),
        stem,
        fill="black"
    )

contact = OUT/"All_New_Figures_Contact_Sheet.jpg"

sheet.save(
    contact,
    quality=90,
    optimize=True
)

print("\nContact sheet generated:", contact.name)

# =============================================================================
# 46K — FINAL MANIFEST AND ZIP FILE
# =============================================================================

manifest = {
    "experiment": (
        "MDAS final manuscript new-figure and "
        "table consolidation"
    ),
    "status": "MANUSCRIPT_FIGURE_TABLE_PACKAGE_COMPLETE",
    "figures":len(figure_inventory_df),
    "table_csv_files":len(list(TABLE_DIR.glob("*.csv"))),
    "png_dpi_for_newly_rendered_figures":900,
    "newly_rendered_figures":[
        "N04","N05"
    ],
    "previously_verified_figures_copied":[
        "N01","N02","N03","N06","N07"
    ],
    "independent_external_acquisitions":1,
    "potsdam_patches":5,
    "perturbations_per_patch":17,
    "source_frozen_records":255,
    "source_posthoc_records":170,
    "maximum_cross_step_difference":(
        maximum_cross_step_difference
    ),
    "registration_estimators_rerun":False,
    "fusion_checkpoints_rerun":False,
    "historical_files_modified":False,
    "figure_removal_decisions_made":False,
    "manuscript_revision_performed":False,
    "source_hashes":{
        "step40_report":sha256(report_paths["step40"]),
        "step41c_report":sha256(report_paths["step41c"]),
        "step42_report":sha256(report_paths["step42"]),
        "step43_report":sha256(report_paths["step43"]),
        "step44c_report":sha256(report_paths["step44c"]),
        "step45_report":sha256(report_paths["step45"]),
        "frozen_results":sha256(FROZEN_CSV),
        "diagnostic_results":sha256(DIAGNOSTIC_CSV),
        "step30_protocol":sha256(PROTOCOL_PATH)
    },
    "limitations":[
        "One independent EnMAP Potsdam acquisition",
        "Five spatially distinct patches from that one scene",
        "Synthetic paired MSI observations",
        "Dependent 17-shift repeated conditions",
        "Post hoc calibration-bias omission diagnostics",
        "No external fusion checkpoint evaluation",
        "Full manuscript must be reviewed before figure removal"
    ],
    "elapsed_seconds":round(time.monotonic()-START,2),
    "output_directory":str(OUT)
}

save_json(
    OUT/"Source_Provenance.json",
    manifest
)

zip_path = OUT.parent / (
    OUT.name + ".zip"
)

if zip_path.exists():
    raise FileExistsError(zip_path)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as z:

    for path in sorted(OUT.rglob("*")):

        if not path.is_file():
            continue

        z.write(
            path,
            arcname=str(
                Path(OUT.name)
                /path.relative_to(OUT)
            )
        )

# Validate ZIP integrity.
with zipfile.ZipFile(zip_path,"r") as z:

    bad = z.testzip()

    if bad is not None:
        raise RuntimeError(
            "ZIP integrity failure: "+bad
        )

print("\n"+"="*90)
print("STEP 46 — FINAL REPORT")
print("="*90)

print(json.dumps({
    "status":manifest["status"],
    "figures":manifest["figures"],
    "tables":manifest["table_csv_files"],
    "zip_file":str(zip_path),
    "output_folder":str(OUT),
    "elapsed_seconds":manifest["elapsed_seconds"]
},indent=2))

print("\nSTEP 46 COMPLETE")
print("Original Augsburg and Potsdam outputs preserved.")
